# 01-Nancang: 数据读入 + 质量控制

## 参数入口（唯一改动处）

本 notebook 的全部可调参数集中在下面四组，运行前只改这里；四组之后进入 preflight 校验与执行，不再散落参数。

### 1. 数据源

In [ ]:
# 数据集 manifest 路径，声明输入格式/路径/raw droplets 路径
MANIFEST_PATH = "data/nancang/manifest.yaml"

### 2. QC 阈值

In [ ]:
# --- QC 策略 ---
# 是什么：QC 阈值策略，adaptive（数据驱动跨样本自适应）或 fixed（固定阈值）
# 默认依据：adaptive，利用数据自身分布自适应调整，比固定阈值更稳健
# 调的影响：fixed 使用固定阈值适合已知标准的数据
# 何时该改：有实验室既定阈值或类器官数据时切 fixed（此时下方四个固定阈值才生效）
QC_STRATEGY = "adaptive"

# --- 自适应 QC 参数 ---
# 是什么：自适应 QC 的 MAD 倍数阈值
# 默认依据：5 为 scRNA-seq 常用宽松界，优先保留更多细胞
# 调大影响：保留更多细胞，可能混入低质量细胞
# 调小影响：更严格过滤，可能误删真实细胞
# 何时该改：数据质量差或双峰分布明显时下调至 3
N_MAD = 5

# 是什么：是否按 sample_id 独立计算 MAD
# 默认依据：True，各样本测序深度和质量不同，独立阈值更公允
# 影响：False 回退全局阈值（兼容旧行为），但可能被高质量样本拉偏整体分布
# 何时该改：样本极少或需与旧结果严格对齐时设 False
PER_SAMPLE_MAD = True

# 是什么：基因保留的最低检出细胞数阈值
# 默认依据：3，去除噪声基因的常用值，平衡真实信号保留与噪声过滤
# 调大影响：更激进去基因，可能丢失稀有细胞类型 marker 基因
# 调小影响：保留更多基因但噪声增多
# 何时该改：关注稀有细胞类型时下调至 1-2
MIN_CELLS_PER_GENE = 3

# --- 固定阈值（仅 QC_STRATEGY="fixed" 时生效）---
# 是什么：fixed 模式下 n_genes 下界，过滤基因数过低的空液滴/死细胞
# 默认依据：200，常规单细胞经验值
# 影响：仅 QC_STRATEGY="fixed" 生效，adaptive 下为占位不参与
# 何时该改：切到 fixed 模式且有明确标准时按需调整
MIN_GENES = 200

# 是什么：fixed 模式下 n_genes 上界，过滤可能的双细胞
# 默认依据：6000，常规单细胞经验值
# 影响：同上，仅 fixed 生效
# 何时该改：高测序深度数据上调
MAX_GENES = 6000

# 是什么：fixed 模式下 total_counts 下界，过滤 UMI 数过低的细胞
# 默认依据：500，常规单细胞经验值
# 影响：同上，仅 fixed 生效
# 何时该改：低深度测序数据下调
MIN_COUNTS = 500

# 是什么：fixed 模式下线粒体百分比上界，过滤细胞膜破损/凋亡细胞
# 默认依据：20%，常规单细胞经验值
# 影响：同上，仅 fixed 生效
# 何时该改：高代谢活性组织（心肌等）上调
MAX_PCT_MT = 20

# --- 血红蛋白标记 ---
# 是什么：血红蛋白基因占比标记阈值（仅标记不删除细胞）
# 默认依据：1.0%，消化道活检中低水平 HB 背景常见
# 调大影响：标记更少细胞，可能漏标红细胞污染
# 调小影响：标记更多细胞，可能误标正常细胞
# 何时该改：红细胞污染疑虑高时下调至 0.5% 复核
HB_THRESHOLD_PCT = 1.0

### 3. 方法开关

In [ ]:
# --- SoupX 环境 RNA 校正 ---
# 是什么：是否启用 SoupX 环境 RNA 校正
# 默认依据：True，nancang 具备完整 raw droplets + filtered 矩阵，是可启用的唯一数据集
# 影响：True 生成 counts_soupx 层供兼容用途，绝不覆盖 counts，失败时对应样本标记 needs_review
# 影响（False）：以未校正 counts 继续（明确选择，非 fallback）
# 何时该改：raw droplets 不可达或 R 不可用时设 False
SOUPX_ENABLED = True

# 是什么：counts_soupx 层是否对校正值取整
# 默认依据：True，部分下游 DEG 方法与离散分布模型要求整数输入
# 影响（False）：保留 SoupX 非整数校正值，适合连续值分析
# 何时该改：下游只用连续值方法或做敏感性分析时设 False
SOUPX_INTEGER_ROUND = True

# --- doublet 检测（决策8：三态定级 singlet/uncertain/doublet）---
# 是什么：scrublet 预期双细胞率
# 默认依据：0.06，10x 常规通量经验值
# 影响：偏高使更多细胞被评为 doublet，None 则完全不运行 doublet 检测
# 何时该改：按上机细胞投入量调整，高负载时上调
EXPECTED_DOUBLET_RATE = 0.06

# 是什么：旧手动双细胞阈值，None 时使用 scrublet 自动阈值
# 默认依据：None，优先用 scrublet 自动阈值，手动指定覆盖自动
# 影响：float 值直接覆盖 scrublet 自动阈值
# 何时该改：自动阈值不合理时手动指定
DOUBLET_SCORE_THRESHOLD = None

# 是什么：三态定级的高置信 doublet 阈值上限
# 默认依据：None，优先用 scrublet 自动阈值 threshold_，其次是 DOUBLET_SCORE_THRESHOLD
# 影响：高于此值标记为 doublet（默认排除）；过高漏检 doublet
# 何时该改：双峰不明显或自动阈值不合理时手动指定
DOUBLET_SCORE_HIGH = None

# 是什么：三态定级的低置信 singlet 上界
# 默认依据：None，自动派生为 HIGH*0.5
# 影响：低于此值标记为 singlet；过低可能误删 singlet
# 何时该改：对纯度要求高时上调 LOW 以扩大 uncertain 区间
DOUBLET_SCORE_LOW = None

# 是什么：uncertain 细胞是否纳入下游分析
# 默认依据：True，只默认排除高置信 doublet，边界细胞标记但保留
# 影响（False）：连 uncertain 一起排除，更保守但损失数据量
# 何时该改：对纯度要求极高时设 False
DOUBLET_UNCERTAIN_INCLUDE = True

# 是什么：单样本 doublet 预测比例上限告警阈值
# 默认依据：0.40，超出此值触发 needs_review 而非静默继续
# 影响：触发 needs_review 要求 PI 审阅 per-sample 诊断表
# 何时该改：不同组织类型预期双率不同时调整
DOUBLET_RATE_ALERT_HIGH = 0.40

# 是什么：单样本 doublet 预测比例下限告警阈值（可选）
# 默认依据：None，不设下限告警
# 影响：None 不触发下限告警，float 值低于此值触发 needs_review
# 何时该改：需要检测异常低 doublet 率时设置
DOUBLET_RATE_ALERT_LOW = None

# 是什么：可稳定计算 doublet 阈值的样本最低细胞数
# 默认依据：50，低于此数阈值估计不可靠，记 needs_review 而非静默跳过
# 影响：低于此数该样本全部标记 singlet（不强行算阈值）
# 何时该改：样本普遍偏小时下调（注意阈值可能不稳）
DOUBLET_MIN_CELLS = 50

# --- 基因标记与评分开关 ---
# 是什么：是否标记血红蛋白基因（仅标记不删除）
# 默认依据：True，消化道活检中 HB 污染信息对下游有参考价值
# 影响（False）：关闭则不产出 flag_hb 列与 pct_counts_hb 指标
# 何时该改：确信无红细胞污染问题时关闭
FLAG_HEMOGLOBIN = True

# 是什么：是否标记应激基因（IEGs + HSPs，组织解离诱导）
# 默认依据：True，解离应激信息对下游质控判断有参考价值
# 影响（False）：关闭则不产出 pct_counts_stress 指标
# 何时该改：不关心解离应激信号时关闭
FLAG_STRESS_GENES = True

# 是什么：是否做细胞周期评分（Tirosh 2015 marker genes）
# 默认依据：True，细胞周期是重要的技术协变量，下游可据此回归
# 影响（False）：关闭则跳过评分 cell，不做周期回归时节省计算
# 何时该改：不做周期回归或数据与周期无关时关闭
SCORE_CELL_CYCLE = True

### 4. 输出版本与运行标识

In [ ]:
# 输出 h5ad 文件的版本号，每次调参建议递增
OUTPUT_VERSION = 1

# 输出文件名，含版本号便于追溯
OUTPUT_FILENAME = "01_nancang_v1.h5ad"

# 运行标识，每次调参改用新 ID，禁止覆盖旧 run 结果
RUN_ID = "01-nancang-v1-run001"

# run 结果存放根目录
RUN_ROOT = "results/runs"

# 随机种子，影响 scrublet 与任何随机过程的可复现性
RANDOM_SEED = 42

In [ ]:
# === Setup：sys.path + 导入依赖 ===
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import subprocess
import shutil
import warnings
import gc
from pathlib import Path
from scipy.stats import median_abs_deviation
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    promote_run, sha256_file, snapshot_effective_parameters, validate_expression_contract,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

from scrna_integration.io import sync_gene_ids
from scrna_integration.platform import check_r_available
RSCRIPT_BIN, R_AVAILABLE = check_r_available()


## Preflight：运行前配置校验

在读入大矩阵、跑 SoupX/scrublet 之前先集中校验路径、输入文件、参数取值与方法开关；任何非法配置在此 raise，避免跑到一半才崩或跑出无法解释的结果。

In [ ]:
# Preflight：运行前配置校验
# 在读入大矩阵之前集中校验路径、输入文件、参数取值与方法开关
# 全部检查 inline 在 cell 内，符合透明性铁律，不进 src
import os, yaml
from pathlib import Path

print("=" * 60)
print("Preflight 校验")
print("=" * 60)

# === A. manifest 与输入文件校验 ===  
# A1. manifest 文件存在性
if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(f"manifest 文件不存在: {MANIFEST_PATH}")
print(f"  [OK] manifest 文件存在: {MANIFEST_PATH}")

# A2. manifest 必需键校验
with open(MANIFEST_PATH) as _pf_f:
    manifest = yaml.safe_load(_pf_f)
for _pf_key in ["input", "source_dataset"]:
    if _pf_key not in manifest:
        raise KeyError(f"manifest 缺少必需键: {_pf_key}")
if "path" not in manifest["input"]:
    raise KeyError("manifest 缺少必需键: input.path")
if "format" not in manifest["input"]:
    raise KeyError("manifest 缺少必需键: input.format")
print(f"  [OK] manifest 必需键完整 (input.path, input.format, source_dataset)")

# A3. 输入数据目录存在性
_pf_input_path = Path(manifest["input"]["path"])
if not _pf_input_path.exists():
    raise FileNotFoundError(f"输入数据目录不存在: {_pf_input_path}")
print(f"  [OK] 输入数据目录存在: {_pf_input_path}")

# A4. 轻量发现校验：确认至少存在一处 10x mtx 文件
_pf_found_mtx = False
# 检查 input.path 直接目录
if (_pf_input_path / "matrix.mtx").exists() or (_pf_input_path / "matrix.mtx.gz").exists():
    _pf_found_mtx = True
# 检查一级子目录
if not _pf_found_mtx:
    for _pf_sub in sorted(_pf_input_path.iterdir()):
        if _pf_sub.is_dir() and not _pf_sub.name.startswith("."):
            if (_pf_sub / "matrix.mtx").exists() or (_pf_sub / "matrix.mtx.gz").exists():
                _pf_found_mtx = True
                break
if not _pf_found_mtx:
    raise FileNotFoundError(
        f"在 {_pf_input_path} 及其一级子目录中未找到任何 matrix.mtx 或 matrix.mtx.gz；"
        f"请确认数据集路径与格式"
    )
print(f"  [OK] 输入目录含 10x mtx 文件（轻量发现通过）")

# === B. 关键 PARAMS 类型/取值合法性校验 ===  
# 逐条 assert，报错信息写清哪个参数、期望什么

# QC 策略
assert QC_STRATEGY in {"adaptive", "fixed"}, \
    f"QC_STRATEGY 必须为 'adaptive' 或 'fixed'，当前值: {QC_STRATEGY!r}"

# 自适应 QC 参数
assert isinstance(N_MAD, (int, float)) and N_MAD > 0, \
    f"N_MAD 必须为正数，当前值: {N_MAD!r}"
assert isinstance(PER_SAMPLE_MAD, bool), \
    f"PER_SAMPLE_MAD 必须为 bool，当前值: {PER_SAMPLE_MAD!r}"
assert isinstance(MIN_CELLS_PER_GENE, int) and MIN_CELLS_PER_GENE >= 1, \
    f"MIN_CELLS_PER_GENE 必须为 >=1 的整数，当前值: {MIN_CELLS_PER_GENE!r}"

# 固定阈值：adaptive 下只校验类型，fixed 下做强校验
for _pf_name, _pf_val in [("MIN_GENES", MIN_GENES), ("MAX_GENES", MAX_GENES),
                           ("MIN_COUNTS", MIN_COUNTS), ("MAX_PCT_MT", MAX_PCT_MT)]:
    assert isinstance(_pf_val, (int, float)) and _pf_val > 0, \
        f"{_pf_name} 必须为正数，当前值: {_pf_val!r}"
if QC_STRATEGY == "fixed":
    assert MIN_GENES < MAX_GENES, \
        f"MIN_GENES({MIN_GENES}) 必须 < MAX_GENES({MAX_GENES})"

# 血红蛋白阈值
assert isinstance(HB_THRESHOLD_PCT, (int, float)) and 0 <= HB_THRESHOLD_PCT <= 100, \
    f"HB_THRESHOLD_PCT 必须为 0-100 的数值，当前值: {HB_THRESHOLD_PCT!r}"

# SoupX 开关
assert isinstance(SOUPX_ENABLED, bool), \
    f"SOUPX_ENABLED 必须为 bool，当前值: {SOUPX_ENABLED!r}"
assert isinstance(SOUPX_INTEGER_ROUND, bool), \
    f"SOUPX_INTEGER_ROUND 必须为 bool，当前值: {SOUPX_INTEGER_ROUND!r}"

# doublet 参数
if EXPECTED_DOUBLET_RATE is not None:
    assert isinstance(EXPECTED_DOUBLET_RATE, (int, float)) and 0 < EXPECTED_DOUBLET_RATE < 1, \
        f"EXPECTED_DOUBLET_RATE 必须为 None 或 0-1 之间的数值，当前值: {EXPECTED_DOUBLET_RATE!r}"
for _pf_name, _pf_val in [("DOUBLET_SCORE_THRESHOLD", DOUBLET_SCORE_THRESHOLD),
                           ("DOUBLET_SCORE_HIGH", DOUBLET_SCORE_HIGH),
                           ("DOUBLET_SCORE_LOW", DOUBLET_SCORE_LOW)]:
    assert _pf_val is None or (isinstance(_pf_val, (int, float)) and _pf_val >= 0), \
        f"{_pf_name} 必须为 None 或 >=0 的数值，当前值: {_pf_val!r}"
if DOUBLET_SCORE_HIGH is not None and DOUBLET_SCORE_LOW is not None:
    assert DOUBLET_SCORE_LOW < DOUBLET_SCORE_HIGH, \
        f"DOUBLET_SCORE_LOW({DOUBLET_SCORE_LOW}) 必须 < DOUBLET_SCORE_HIGH({DOUBLET_SCORE_HIGH})"
assert isinstance(DOUBLET_UNCERTAIN_INCLUDE, bool), \
    f"DOUBLET_UNCERTAIN_INCLUDE 必须为 bool，当前值: {DOUBLET_UNCERTAIN_INCLUDE!r}"
assert isinstance(DOUBLET_RATE_ALERT_HIGH, (int, float)) and 0 < DOUBLET_RATE_ALERT_HIGH <= 1, \
    f"DOUBLET_RATE_ALERT_HIGH 必须为 0-1 之间的数值，当前值: {DOUBLET_RATE_ALERT_HIGH!r}"
if DOUBLET_RATE_ALERT_LOW is not None:
    assert isinstance(DOUBLET_RATE_ALERT_LOW, (int, float)) and 0 <= DOUBLET_RATE_ALERT_LOW < DOUBLET_RATE_ALERT_HIGH, \
        f"DOUBLET_RATE_ALERT_LOW({DOUBLET_RATE_ALERT_LOW}) 必须 < DOUBLET_RATE_ALERT_HIGH({DOUBLET_RATE_ALERT_HIGH})"
assert isinstance(DOUBLET_MIN_CELLS, int) and DOUBLET_MIN_CELLS > 0, \
    f"DOUBLET_MIN_CELLS 必须为正整数，当前值: {DOUBLET_MIN_CELLS!r}"

# 运行标识
assert isinstance(RANDOM_SEED, int), \
    f"RANDOM_SEED 必须为 int，当前值: {RANDOM_SEED!r}"
assert isinstance(OUTPUT_VERSION, int), \
    f"OUTPUT_VERSION 必须为 int，当前值: {OUTPUT_VERSION!r}"

# 标记与评分开关
for _pf_name, _pf_val in [("FLAG_HEMOGLOBIN", FLAG_HEMOGLOBIN),
                           ("FLAG_STRESS_GENES", FLAG_STRESS_GENES),
                           ("SCORE_CELL_CYCLE", SCORE_CELL_CYCLE)]:
    assert isinstance(_pf_val, bool), \
        f"{_pf_name} 必须为 bool，当前值: {_pf_val!r}"

# === C. SoupX 开关与数据集匹配校验 ===  
# nancang 是三个数据集中唯一有 SoupX 的，preflight 校验最复杂
if SOUPX_ENABLED is True:
    # C1. manifest 中 raw_path 键存在性
    _pf_raw_path = manifest["input"].get("raw_path")
    if _pf_raw_path is None:
        raise KeyError(
            "SOUPX_ENABLED=True 但 manifest 未声明 input.raw_path；"
            "请补 raw droplets 路径或设 SOUPX_ENABLED=False"
        )
    # C2. raw droplets 目录存在性
    _pf_raw_dir = Path(_pf_raw_path)
    if not _pf_raw_dir.exists():
        raise FileNotFoundError(
            f"SOUPX_ENABLED=True 但 raw droplets 目录不存在: {_pf_raw_dir}"
        )
    print(f"  [OK] SoupX raw droplets 目录: {_pf_raw_dir}")
    
    # C3. SoupX R 脚本可达性
    if not os.path.exists("scripts/soupx_run.R"):
        raise FileNotFoundError(
            "SOUPX_ENABLED=True 但 scripts/soupx_run.R 不可达；"
            "请确认该脚本在项目根 scripts/ 目录下"
        )
    print(f"  [OK] SoupX R 脚本可访问: scripts/soupx_run.R")
    
    # C4. R 环境状态（graceful：R 不可用不 raise，只 WARNING）
    # SoupX cell 对 R 缺失是 graceful skip（status=skipped），preflight 与之一致
    if R_AVAILABLE is False:
        print(f"  [WARNING] SOUPX_ENABLED=True 但 R 环境未就绪；"
              f"SoupX cell 将按既定逻辑跳过（soupx_contract.status=skipped），"
              f"doublet 将回退到 layers['counts'] 输入")
    else:
        print(f"  [OK] R 环境就绪: {RSCRIPT_BIN}")
else:
    print(f"  [INFO] SoupX 已按配置关闭，将以 layers['counts'] 继续（属明确选择）")

# === D. doublet 方法开关提示（不 raise，只 print 生效方法）===  
if EXPECTED_DOUBLET_RATE is not None:
    print(f"  [INFO] doublet 方法: scrublet per-sample（expected_rate={EXPECTED_DOUBLET_RATE}）")
else:
    print(f"  [INFO] doublet 未运行（EXPECTED_DOUBLET_RATE=None，全部标记 singlet）")

# === E. preflight 汇总打印 ===  
print()
print("-" * 60)
print("Preflight 校验通过，当前生效参数:")
print(f"  manifest      : {MANIFEST_PATH}")
print(f"  input.path    : {manifest['input']['path']}")
print(f"  QC_STRATEGY   : {QC_STRATEGY}")
print(f"  N_MAD          : {N_MAD}")
print(f"  PER_SAMPLE_MAD : {PER_SAMPLE_MAD}")
print(f"  SOUPX_ENABLED  : {SOUPX_ENABLED}")
if SOUPX_ENABLED and R_AVAILABLE:
    print(f"  SoupX raw_path : {manifest['input'].get('raw_path', 'N/A')}")
print(f"  SOUPX_INTEGER_ROUND: {SOUPX_INTEGER_ROUND}")
print(f"  EXPECTED_DOUBLET_RATE: {EXPECTED_DOUBLET_RATE}")
print(f"  RANDOM_SEED    : {RANDOM_SEED}")
print(f"  OUTPUT_VERSION : {OUTPUT_VERSION}")
print(f"  R 环境          : {'就绪' if R_AVAILABLE else '未就绪（SoupX 将跳过）'}")
print("-" * 60)

# 清理 preflight 临时变量，避免污染 notebook 全局命名空间
# 逐个 try/except 清理，因为部分变量仅在特定代码路径定义
# （_pf_sub 在 for 循环分支内、_pf_f 在 with 块内）
for _pf_var in ['_pf_f', '_pf_key', '_pf_name', '_pf_val', '_pf_input_path',
                 '_pf_found_mtx', '_pf_sub', '_pf_raw_path', '_pf_raw_dir', 'manifest']:
    try:
        exec(f'del {_pf_var}')
    except NameError:
        pass

## 回跑与新版本

### 何时需要回跑

- 调整 QC 阈值（`N_MAD`、`MIN_CELLS_PER_GENE` 等）
- 改变方法开关（`SOUPX_ENABLED`、`DOUBLET_DETECTION`、`FLAG_HEMOGLOBIN` 等）
- 更新 `EXPECTED_DOUBLET_RATE` 或双细胞阈值参数
- SoupX 校正参数调整（需要重新运行 R 端 SoupX）
- 上游临床元数据更新
- 数据文件路径变更

### 调参回跑步骤

1. **bump `OUTPUT_VERSION`**：在 PARAMS cell 中将 `OUTPUT_VERSION` 从 `"v1"` 改为 `"v2"`（`RUN_ID` 自动更新为 `"01-nancang-v2"`）
2. 修改需调整的参数
3. 从头运行全部 cell（Kernel → Restart & Run All）
4. 新 run 写入独立目录，旧 run 结果完整保留、不会被覆盖
5. 确认 Stage 01 Verdict 检查清单全部通过

### RUN_ID 与目录结构

每个 `RUN_ID` 在 `results/runs/` 下创建独立目录，版本间互不覆盖：

```
results/runs/
  01-nancang-v1/
    draft/manifest.json       # 执行事实验证记录
    draft/01_nancang_v1.h5ad  # checkpoint 产物
    promoted/                  # promote 后出现（SUCCESS 状态）
  01-nancang-v2/
    draft/
    ...
```

### 上下游关系

- **上游**：cellranger 输出的 `filtered_feature_bc_matrix` 目录（10x mtx 格式）
- **下游**：`02_merged.ipynb` 读取所有 per-dataset h5ad 进行跨数据集整合
- **契约**：产出的 h5ad 必须通过 `validate_per_dataset_output()` 校验才能进入 merge

### 常见问题

- **RUN_ID 已存在**：`prepare_run()` 会自动检测并提示冲突；确认要覆盖则手动删除旧目录后重跑
- **NEEDS_REVIEW**：双细胞检测或 SoupX 异常时触发，检查 per-sample 诊断表后 PI 决定是否手动 promote 或调整参数重跑
- **schema 校验失败**：检查 `layers["counts"]` 是否已建立、`doublet_class` 是否为 Categorical、`expression_contract` 是否完整
- **SoupX 失败**：检查 R 环境是否就绪（`Rscript --version`），确认 raw_feature_bc_matrix 路径正确

In [ ]:
# 数据读入：10x mtx 格式，多子目录发现
# 替代原来的 read_with_manifest / 按透明性铁律，数据读取逻辑拆回 cell
import yaml, anndata
from pathlib import Path
from scrna_integration.io import sync_gene_ids

with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)
source_dataset = str(manifest["source_dataset"])

data_dir = Path(manifest["input"]["path"])
assert data_dir.exists(), f"数据目录不存在: {data_dir}"

# 发现所有 10x mtx 子目录（每个含 matrix.mtx(.gz)）
sub_dirs = []
for sub in sorted(data_dir.iterdir()):
    if sub.is_dir() and not sub.name.startswith("."):
        if (sub / "matrix.mtx").exists() or (sub / "matrix.mtx.gz").exists():
            sub_dirs.append(sub)
if not sub_dirs:
    if (data_dir / "matrix.mtx").exists() or (data_dir / "matrix.mtx.gz").exists():
        sub_dirs.append(data_dir)
print(f"发现 {len(sub_dirs)} 个 10x 数据子目录: {[d.name for d in sub_dirs]}")

# 逐个子目录读取，拼接为完整 AnnData
adatas = []
for sub_dir in sub_dirs:
    adata_sub = sc.read_10x_mtx(sub_dir, var_names="gene_symbols")
    adata_sub.obs_names = [f"{sub_dir.name}_{bc}" for bc in adata_sub.obs_names]
    adata_sub.obs["source_dataset"] = source_dataset
    adatas.append(adata_sub)
    print(f"  {sub_dir.name}: {adata_sub.n_obs:,} 细胞 x {adata_sub.n_vars:,} 基因")

# 多子目录拼接（单目录直接使用）
if len(adatas) > 1:
    adata = anndata.concat(adatas, join="outer", index_unique="_")
else:
    adata = adatas[0]

# 基因 ID 同步：var.index 为 symbol → 补充 var["ensembl_id"] 列
# 这是 02_merged inner join 的前提——所有数据集的 var 含统一的 ensembl_id
sync_gene_ids(adata, gene_id_format="symbol")

print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")


In [ ]:
# === expression_contract：建立 counts 契约（决策1/2） ===
# 目的：Nancang 的 X 为 raw counts（10x mtx 格式），建立 layers["counts"] 作为唯一权威计数层。
# layers["counts"] 是进入框架后唯一 counts 权威位置，一经建立不得被覆盖。
# SoupX 属 P1，本轮 soupx_layer 固定为 None。

# 1. 建立 layers["counts"] = CSR float32，从 X 复制
adata.layers["counts"] = sp.csr_matrix(adata.X, dtype=np.float32)
print(f"layers['counts'] 已建立: shape={adata.layers['counts'].shape}, "
      f"dtype={adata.layers['counts'].dtype}, format={adata.layers['counts'].format}")

# 2. 全量验证 counts 为非负整数（raw counts 准入条件）
# 稀疏矩阵 .data 包含全部非零值，零值天然通过整数检查
_counts_data = adata.layers["counts"].data
_counts_are_nonneg = bool(np.all(_counts_data >= 0))
_counts_are_int = bool(np.all(_counts_data == np.floor(_counts_data)))
_counts_ok = bool(_counts_are_nonneg and _counts_are_int)
if not _counts_are_nonneg:
    raise ValueError(f"counts 包含负值：min={_counts_data.min():.1f}，非 raw counts")
if not _counts_are_int:
    raise ValueError(f"counts 包含非整数值，非 raw counts")
_min_count = float(_counts_data.min())
_max_count = float(_counts_data.max())

# 3. 建立 expression_contract（8 字段 schema，见 docs/整改执行分解-P0-P3-20260714.md 第一节）
adata.uns["expression_contract"] = {
    "x_scale": "raw_counts",          # X 当前尺度：raw_counts
    "counts_layer": "counts",         # 权威原始 counts 所在 layer
    "counts_source": "X",             # counts 直接从 X 提取（10x mtx 的 X 就是 raw）
    "counts_validated": _counts_ok,   # 非负整数校验结果
    "counts_integer_check": "full",   # 全量整数校验（非 blockwise）
    "soupx_layer": None,              # SoupX 属 P1，本轮不动
    "processing_history": [           # 逐步处理记录
        "01_nancang: loaded from 10x mtx, X is raw integer counts, "
        "copied to layers[counts] (CSR float32)"
    ],
    "stage": "01",                    # Stage 01 产物
}

# 4. 校验契约 schema（8 字段完整性 + 取值合法性）
_contract = validate_expression_contract(adata, expected_scale="raw_counts", stage="01")
print(f"\nexpression_contract 通过校验:")
print(f"  x_scale             = {_contract['x_scale']}")
print(f"  counts_layer        = {_contract['counts_layer']}")
print(f"  counts_source       = {_contract['counts_source']}")
print(f"  counts_validated    = {_contract['counts_validated']}")
print(f"  counts_integer_check= {_contract['counts_integer_check']}")
print(f"  soupx_layer         = {_contract['soupx_layer']}")
print(f"  stage               = {_contract['stage']}")
print(f"  count range: [{_min_count}, {_max_count}]")


# SoupX 不覆盖 counts 的可执行门禁（决策3 红线）：保存 checksum，checkpoint 验证 layers["counts"] 全程未改
# 在 SoupX cell 之前记录，checkpoint cell 断言不变——这是"counts只读不改"的唯一机械门禁
_counts_checksum = float(adata.layers["counts"].sum())
_counts_checksum_nnz = int(adata.layers["counts"].nnz)
del _counts_data, _counts_are_nonneg, _counts_are_int, _counts_ok, _min_count, _max_count


## 样本级 QC 摘要

按 sample_id 分组统计每个样本的细胞数、基因中位数、UMI 中位数、线粒体比例中位数。
**看什么**：是否存在某个样本与其他样本差异过大（如某样本细胞数极少、或 MT% 异常偏高）。
这将帮助判断是否需要为特定样本设置差异化阈值。

In [ ]:
# 样本级 QC 摘要表
print("===== 样本级 QC 摘要 =====")
sample_summary = adata.obs.groupby("sample_id").agg(
    n_cells=("n_genes", "count"),
    median_n_genes=("n_genes", "median"),
    median_total_counts=("total_counts", "median"),
    median_pct_mt=("pct_counts_mt", "median"),
).sort_values("n_cells", ascending=False)
display(sample_summary)

abnormal = sample_summary[
    (sample_summary["n_cells"] < 100) | (sample_summary["median_pct_mt"] > 30)
]
if len(abnormal) > 0:
    print("\nWARNING 异常样本（n_cells<100 或 median_pct_mt>30%）：")
    display(abnormal)
else:
    print("\n所有样本通过初步检查。")


## 基线 QC 分布

绘制三个主 QC 指标的小提琴图和散点图，供 PI 在设定过滤阈值前直观判断数据质量。

**三个指标的含义**：
- `n_genes`：每个细胞检测到的基因数。过低→空液滴或死细胞；过高→可能是双细胞
- `total_counts`：每个细胞的总 UMI 计数。分布应与 n_genes 正相关
- `pct_counts_mt`：线粒体转录本百分比。过高（>20%）→细胞膜破损/凋亡

**如何使用这些图**：
1. 先看小提琴图，了解各指标的总体分布范围和离群情况
2. 再看散点图，检查 n_genes vs pct_mt 的关系——通常呈负相关
3. 根据分布特征，回到顶部 PARAMS 调整 N_MAD 或固定阈值

In [ ]:
# QC 小提琴图：按 sample_id 分组展示三个主指标。
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="sample_id", rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤前）")
plt.tight_layout()
fig.savefig("results/figures/01_nancang_qc_violin_pre.png", dpi=150, bbox_inches="tight")
plt.show()

# QC 散点图
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤前）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts", ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤前）")
plt.tight_layout()
fig.savefig("results/figures/01_nancang_qc_scatter_pre.png", dpi=150, bbox_inches="tight")
plt.show()


## 自适应阈值计算（MAD-based）

MAD（median absolute deviation）是比标准差更稳健的离散度度量，对离群值不敏感。

**参数含义**：
- `N_MAD = 5`：阈值 = 中位数 +/- N_MAD * MAD
- 越大越宽松（保留更多细胞），越小越严格（去除更多细胞）
- `n_genes` 做双侧过滤（过低+过高），`total_counts` 仅下界，`pct_counts_mt` 仅上界

**Why MAD 而不是固定阈值**：不同数据集/样本的 baseline 差异很大（如组织活检 vs 类器官的 MT% 基线差异可达 3-5 倍）。
MAD 基于每个数据集自身的分布自适应调整，避免用一个固定阈值削足适履。

In [ ]:
# 自适应阈值计算（MAD-based）
n_before = adata.n_obs

# 过滤前统计摘要
qc_pre_stats = {
    "n_genes": {"median": float(adata.obs["n_genes"].median()), "mean": float(adata.obs["n_genes"].mean())},
    "total_counts": {"median": float(adata.obs["total_counts"].median()), "mean": float(adata.obs["total_counts"].mean())},
    "pct_counts_mt": {"median": float(adata.obs["pct_counts_mt"].median()), "mean": float(adata.obs["pct_counts_mt"].mean())},
}

if QC_STRATEGY == "adaptive":
    thresholds = {}
    for metric, direction in [("n_genes", "both"), ("total_counts", "lower"), ("pct_counts_mt", "upper")]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组，每个样本独立计算阈值
            sample_thresholds = {}
            for sample_id in adata.obs["sample_id"].unique():
                mask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[mask, metric].dropna()
                if len(vals) < 10:
                    print(f"  WARNING: {sample_id} 仅有 {len(vals)} 个细胞的 {metric} 值，沿用全局阈值")
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                lower = max(0, med - N_MAD * mad_val) if direction in ("both", "lower") else None
                upper = med + N_MAD * mad_val if direction in ("both", "upper") else None
                sample_thresholds[sample_id] = {"median": round(med, 1), "mad": round(mad_val, 1),
                                                 "lower": round(lower, 1) if lower else None,
                                                 "upper": round(upper, 1) if upper else None}
            thresholds[metric] = {"mode": "per_sample", "per_sample": sample_thresholds}
        else:
            # 全局 MAD（原逻辑）
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad = median_abs_deviation(vals, nan_policy="omit")
            lower = max(0, med - N_MAD * mad) if direction in ("both", "lower") else None
            upper = med + N_MAD * mad if direction in ("both", "upper") else None
            thresholds[metric] = {"median": round(med, 1), "mad": round(mad, 1),
                                  "lower": round(lower, 1) if lower else None,
                                  "upper": round(upper, 1) if upper else None}
elif QC_STRATEGY == "fixed":
    # 固定阈值模式：适用于已知阈值的数据集（如类器官、实验室内部标准）
    thresholds = {
        "n_genes": {"median": None, "mad": None, "lower": MIN_GENES if 'MIN_GENES' in dir() else 200, "upper": MAX_GENES if 'MAX_GENES' in dir() else 6000},
        "total_counts": {"median": None, "mad": None, "lower": MIN_COUNTS if 'MIN_COUNTS' in dir() else 500, "upper": None},
        "pct_counts_mt": {"median": None, "mad": None, "lower": None, "upper": MAX_PCT_MT if 'MAX_PCT_MT' in dir() else 20},
    }
    print("使用固定阈值模式")
else:
    raise ValueError(f"不支持的 QC_STRATEGY: {QC_STRATEGY}，请使用 'adaptive' 或 'fixed'")

print(f"===== QC 阈值（{QC_STRATEGY}）=====")
print(f"  策略: {QC_STRATEGY}")
if QC_STRATEGY == "adaptive":
    print(f"  N_MAD: {N_MAD}")
    if PER_SAMPLE_MAD:
        print(f"  模式: per-sample（每个 sample_id 独立计算 MAD）")
    else:
        print(f"  模式: global（全局 MAD）")

if PER_SAMPLE_MAD:
    # 打印每个 sample 的阈值汇总表
    print("\n===== 各样本阈值明细 =====")
    for metric in ["n_genes", "total_counts", "pct_counts_mt"]:
        print(f"\n--- {metric} ---")
        rows = []
        for sid, t in thresholds[metric]["per_sample"].items():
            rows.append({"sample_id": sid, **{k: v for k, v in t.items() if v is not None}})
        if rows:
            display(pd.DataFrame(rows).set_index("sample_id"))
else:
    thresh_df = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in thresholds.items()}).T
    display(thresh_df)

In [ ]:
# --- 跨样本阈值 forest plot（仅 per-sample 模式）---
# 展示各样本的 MAD 阈值范围，可直观比较各样本的 QC 特性差异
if PER_SAMPLE_MAD and QC_STRATEGY == "adaptive":
    fig, axes = plt.subplots(1, 3, figsize=(15, max(4, len(adata.obs["sample_id"].unique()) * 0.4)))
    for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
        ax = axes[i]
        per_sample = thresholds[metric].get("per_sample", {})
        samples = sorted(per_sample.keys())
        if not samples:
            ax.set_title(f"{metric}\n（无 per-sample 数据）")
            continue
        y_pos = range(len(samples))
        lowers = [per_sample[s].get("lower", 0) or 0 for s in samples]
        uppers = [per_sample[s].get("upper", 0) or 0 for s in samples]
        medians = [per_sample[s].get("median", 0) for s in samples]
        # 用上界-下界的水平 bar 展示阈值区间
        ax.barh(y_pos, [u - l for u, l in zip(uppers, lowers)], left=lowers, height=0.6,
                alpha=0.3, color="steelblue")
        ax.scatter(medians, y_pos, color="red", zorder=5, s=20, label="median")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(samples, fontsize=8)
        ax.set_xlabel(metric)
        ax.set_title(f"{metric} 阈值范围")
        if i == 0:
            ax.legend(fontsize=7, loc="lower right")
    plt.suptitle("Per-sample MAD 阈值 Forest Plot", fontsize=12)
    plt.tight_layout()
    plt.savefig("results/figures/01_nancang_qc_forest.png", dpi=150, bbox_inches="tight")
    plt.show()
elif PER_SAMPLE_MAD and QC_STRATEGY == "fixed":
    print("fixed 模式下无 per-sample 阈值，跳过 forest plot")


## N_MAD 敏感度分析

核心问题：N_MAD 太小 -> 丢太多细胞（可能丢真信号）；太大 -> 保留垃圾。
经验法则：组织活检 3-4，类器官 5-7。下面的曲线帮助你选择最佳值。


In [ ]:
# === N_MAD 敏感度分析：帮助 PI 选择最优阈值 ===
# F4修复：敏感度曲线与过滤同口径——PER_SAMPLE_MAD 开关同时作用于曲线与过滤
# 此前曲线始终使用全局 MAD，PI 据曲线选参数会被误导
_test_mads = sorted(set([2, 3, 4, 5, 6, 7, N_MAD, N_MAD + 1]))
sensitivity_results = []
for _nm in _test_mads:
    _keep = pd.Series(True, index=adata.obs_names)
    for metric, direction in [("n_genes", "both"), ("total_counts", "lower"), ("pct_counts_mt", "upper")]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组独立算 MAD（与阈值计算 + 实际过滤同口径）
            for sample_id in adata.obs["sample_id"].unique():
                smask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[smask, metric].dropna()
                # 样本细胞数不足 10，无法可靠估计 MAD，该样本细胞不参与本维度过滤
                if len(vals) < 10:
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                if direction in ("both", "lower"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] >= max(0, med - _nm * mad_val)
                if direction in ("both", "upper"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] <= med + _nm * mad_val
        else:
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad_val = median_abs_deviation(vals, nan_policy="omit")
            if direction in ("both", "lower"):
                _keep &= adata.obs[metric] >= max(0, med - _nm * mad_val)
            if direction in ("both", "upper"):
                _keep &= adata.obs[metric] <= med + _nm * mad_val
    n_keep = _keep.sum()
    sensitivity_results.append({
        "N_MAD": _nm,
        "cells_kept": n_keep,
        "pct_kept": round(100 * n_keep / adata.n_obs, 1),
        "median_mt_kept": round(adata.obs.loc[_keep, "pct_counts_mt"].median(), 2),
    })

sens_df = pd.DataFrame(sensitivity_results)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sens_df["N_MAD"], sens_df["pct_kept"], "o-", color="steelblue", linewidth=2)
ax1.axvline(N_MAD, color="red", linestyle="--", label=f"当前 N_MAD={N_MAD}")
ax1.set_xlabel("N_MAD")
ax1.set_ylabel("细胞保留率 (%)", color="steelblue")
ax1.set_title("N_MAD 敏感度：保留率 vs 阈值宽松度")
ax2 = ax1.twinx()
ax2.plot(sens_df["N_MAD"], sens_df["median_mt_kept"], "s--", color="orange")
ax2.set_ylabel("保留细胞的 median MT%", color="orange")
ax1.legend()
plt.tight_layout()
plt.savefig("results/figures/01_nancang_mad_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print(sens_df.to_string(index=False))
print("\n经验判据：保留 85-95% 细胞 + median MT% 不显著上升 = 合理")


In [ ]:
# ── 各样本 MAD 敏感度表 ──────────────────────────────
# 对每个 N_MAD 值，分别计算各样本的 QC 过滤保留率
# 帮助识别：某 N_MAD 下哪个样本损失最多细胞（可能是技术异常样本）
# 注意：此表仅用于诊断，不改变上方 PARAMS 的 N_MAD 设置

_sample_retention = {}
for _m in _test_mads:
    _keep = {}
    for _s, _grp in adata.obs.groupby("sample_id"):   # 按样本分组
        _n_total = len(_grp)
        # 计算在此 N_MAD 下每个 QC 指标的 MAD 阈值
        _mt_med = _grp["pct_counts_mt"].median()
        _mt_mad = median_abs_deviation(_grp["pct_counts_mt"])
        _gene_med = _grp["n_genes"].median()
        _gene_mad = median_abs_deviation(_grp["n_genes"])
        _mask = (
            (_grp["pct_counts_mt"] <= _mt_med + _m * _mt_mad) &
            (_grp["n_genes"] >= _gene_med - _m * _gene_mad)
        )
        _keep[_s] = f"{_mask.sum()} / {_n_total}  ({100*_mask.mean():.1f}%)"
    _sample_retention[f"N_MAD={_m}"] = _keep

import pandas as pd
_df_retention = pd.DataFrame(_sample_retention)
# 高亮当前 N_MAD 列
print(f"\n── 各样本在不同 N_MAD 下的细胞保留率 ──")
print(f"（当前设置：N_MAD = {N_MAD}，对应列已标注 ← ）")
_cols = [c + (" ←" if f"={N_MAD}" in c else "") for c in _df_retention.columns]
_df_retention.columns = _cols
display(_df_retention)


In [ ]:
# ── N_MAD 诊断建议 ───────────────────────────────────
# 根据各样本间 QC 指标的变异系数（CV）给出参考建议
# CV 高 → 样本间技术差异大 → 建议使用较宽 MAD（避免因个别高质量/低质量样本扭曲阈值）
# CV 低 → 样本技术质量均一 → 可以使用较严 MAD
# 注意：此建议仅供参考，最终 N_MAD 由 PI 结合生物学背景决定

from scipy.stats import variation  # 无新依赖：scipy 已在环境中

_per_sample_stats = adata.obs.groupby("sample_id")[
    ["n_genes", "pct_counts_mt"]
].median()

_cv_genes = variation(_per_sample_stats["n_genes"])
_cv_mt = variation(_per_sample_stats["pct_counts_mt"].replace(0, 1e-6))

print(f"\n── N_MAD 诊断建议 ──────────────────────────────")
print(f"各样本间 n_genes  变异系数（CV）= {_cv_genes:.3f}")
print(f"各样本间 pct_mt   变异系数（CV）= {_cv_mt:.3f}")

_cv_max = max(_cv_genes, _cv_mt)
if _cv_max < 0.1:
    _suggestion = "3-4（样本质量均一，可用较严阈值）"
elif _cv_max < 0.25:
    _suggestion = "4-5（样本间有中等差异，推荐默认区间）"
else:
    _suggestion = "5-7（样本间差异较大，建议宽松阈值避免过度过滤）"

print(f"\n推荐 N_MAD 区间：{_suggestion}")
# 检查当前 N_MAD 是否在推荐区间内
_sug_range = _suggestion[:3].split("-")
_in_range = any(int(x) == N_MAD for x in _sug_range)
print(f"当前设置：N_MAD = {N_MAD}  {'✓ 在推荐范围内' if _in_range else '← 请结合上方保留率表评估是否需要调整'}")
print(f"\n使用方式：若需调整，修改顶部 PARAMS cell 中的 N_MAD 后重跑本 notebook。")
print(f"────────────────────────────────────────────────")


## 基因复杂度

`log_complexity = log10(n_genes+1) / log10(total_counts+1)` 反映每个细胞的"基因多样性密度"。
复杂度异常低（相同 UMI 下基因数过少）提示该细胞可能只捕获了极少数高表达基因，
是低质量细胞的补充判据。

In [ ]:
# 基因复杂度
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(adata.obs["log_complexity"].dropna(), bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("log10(n_genes+1) / log10(total_counts+1)")
ax.set_ylabel("细胞数")
ax.set_title("基因复杂度分布")
for pct, color in [(1, "red"), (5, "orange"), (50, "green"), (95, "orange"), (99, "red")]:
    val = np.percentile(adata.obs["log_complexity"].dropna(), pct)
    ax.axvline(val, color=color, linestyle="--", alpha=0.5, linewidth=0.8)
plt.tight_layout()
fig.savefig("results/figures/01_nancang_complexity.png", dpi=150, bbox_inches="tight")
plt.show()
for pct in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  P{pct}: {np.percentile(adata.obs['log_complexity'].dropna(), pct):.4f}")


## 特殊基因标记

**血红蛋白基因（HB）**：红细胞污染/溶解的标志。仅标记不移除——消化道活检中低水平 HB 背景常见。
**应激基因**：即时早期基因（IEGs）+ 热休克蛋白（HSPs），在组织解离过程中被机械/酶切应激诱导。
标记供下游分析参考，不是过滤依据。

In [ ]:
# 血红蛋白基因标记（仅标记，不移除）
if FLAG_HEMOGLOBIN:
    adata.var["hb"] = adata.var_names.str.upper().str.match("^HB[^P]")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["hb"], percent_top=None, log1p=False, inplace=True)
    adata.obs["flag_hb"] = adata.obs["pct_counts_hb"] > HB_THRESHOLD_PCT
    n_hb_flagged = int(adata.obs["flag_hb"].sum())
    print(f"血红蛋白标记: {n_hb_flagged} / {adata.n_obs} 细胞 pct_hb > {HB_THRESHOLD_PCT}%  ({100*n_hb_flagged/adata.n_obs:.1f}%)")
else:
    print("⏭ 跳过血红蛋白标记（FLAG_HEMOGLOBIN=False）")

# 应激基因标记（仅标记，不移除）
if FLAG_STRESS_GENES:
    STRESS_GENES = ['JUN', 'FOS', 'JUNB', 'FOSB', 'ATF3', 'HSPA1A', 'HSPA1B', 'HSP90AA1', 'HSP90AB1', 'DNAJB1', 'HSPB1']
    stress_in_data = [g for g in STRESS_GENES if g in adata.var_names]
    if stress_in_data:
        adata.var["stress"] = adata.var_names.isin(stress_in_data)
        sc.pp.calculate_qc_metrics(adata, qc_vars=["stress"], percent_top=None, log1p=False, inplace=True)
        print(f"应激基因: {len(stress_in_data)}/{len(STRESS_GENES)} 个检测到, 中位 pct={adata.obs['pct_counts_stress'].median():.2f}%")
    else:
        print("WARNING: 数据中未检测到应激基因")
else:
    print("⏭ 跳过应激基因标记（FLAG_STRESS_GENES=False）")

## 环境 RNA 校正（SoupX）

**决策3**：校正结果写入 `layers["counts_soupx"]`（CSR float32），绝不覆盖 `layers["counts"]`（唯一权威原始计数层）或 `X`。
`counts` 只读不改，SoupX 校正结果只支持兼容用途，不进 scVI。

环境 RNA（ambient RNA）来自裂解的细胞碎片和游离 RNA，悬浮在液滴溶液中并被随机捕获形成背景噪声。
SoupX 利用 raw matrix（含空液滴/碎片背景）和 filtered matrix（只含真实细胞）之间的差异，
估计每个基因的污染比例并扣除。

**为什么用 subprocess Rscript 而不是 rpy2**：
rpy2 + anndata2ri 在 conda R 4.4.3 下存在严重兼容性问题。subprocess 独立进程通过临时 mtx 文件交换数据，进程隔离避免桥接崩溃。

**三重守卫**（任一不满足则优雅跳过）：
1. `SOUPX_ENABLED=True`
2. Manifest 声明了 `raw_matrix_path`
3. Rscript 可执行 + SoupX R 包可加载

In [ ]:
# 环境 RNA 校正（SoupX）—— subprocess Rscript 模式
# 决策3：校正结果写入 layers["counts_soupx"]，绝不覆盖 layers["counts"]（唯一权威原始计数层）
# counts 是只读基线，SoupX 校正结果只支持兼容用途，不进 scVI
# 决策3+9：status.json 双重判定——R exit code + json status!=success 任一即判定该样本失败
# 决策4：SOUPX_INTEGER_ROUND=True 对校正值取整，满足部分下游 DEG 方法对整数输入的要求

soupx_applied = False
n_soupx_corrected = 0
soupx_needs_review = False  # 启用但部分/全部样本失败→需 PI 审视
soupx_failed_samples = []
soupx_diagnostics = {}

# 初始化 counts_soupx layer：从原始 counts 拷贝作基线；未校正的样本/细胞保持原始 counts 值
# 在循环前全部初始化，避免 per-sample 渐进式写入导致的内存碎片
if "counts_soupx" not in adata.layers:
    adata.layers["counts_soupx"] = adata.layers["counts"].copy()
    # 确保 CSR float32（内存纪律：稀疏 + 32位浮点，不 densify）
    if not sp.issparse(adata.layers["counts_soupx"]) or adata.layers["counts_soupx"].dtype != np.float32:
        adata.layers["counts_soupx"] = sp.csr_matrix(adata.layers["counts_soupx"], dtype=np.float32)
    print(f"layers['counts_soupx'] 已初始化: shape={adata.layers['counts_soupx'].shape}, "
          f"dtype={adata.layers['counts_soupx'].dtype}, nnz={adata.layers['counts_soupx'].nnz}")

if not SOUPX_ENABLED:
    print(f"SoupX 已跳过: SOUPX_ENABLED=False。")
    # 跳过分支也要写 soupx_contract（保证下游恒可读取）
    adata.uns["soupx_contract"] = {
        "enabled": False, "status": "skipped",
        "n_cells_corrected": 0, "integer_round": SOUPX_INTEGER_ROUND,
        "per_sample": {}, "failed_samples": [], "needs_review": False,
    }
elif not adata.uns.get("raw_matrix_path"):
    print("SoupX 已跳过: adata.uns 中无 raw_matrix_path。")
    adata.uns["soupx_contract"] = {
        "enabled": True, "status": "skipped",
        "n_cells_corrected": 0, "integer_round": SOUPX_INTEGER_ROUND,
        "per_sample": {}, "failed_samples": [], "needs_review": False,
        "skip_reason": "no raw_matrix_path in manifest",
    }
elif not R_AVAILABLE:
    print("SoupX 已跳过: R 环境未就绪。")
    adata.uns["soupx_contract"] = {
        "enabled": True, "status": "skipped",
        "n_cells_corrected": 0, "integer_round": SOUPX_INTEGER_ROUND,
        "per_sample": {}, "failed_samples": [], "needs_review": False,
        "skip_reason": "R not available",
    }
else:
    raw_path = adata.uns["raw_matrix_path"]
    soupx_script = "scripts/soupx_run.R"
    soupx_tmp = "results/_soupx_tmp"
    os.makedirs(soupx_tmp, exist_ok=True)
    import scipy.io

    print(f"raw_matrix_path: {raw_path}")
    if "ambient_correction_applied" not in adata.obs.columns:
        adata.obs["ambient_correction_applied"] = False

    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_cells = sample_mask.sum()
        cell_ids = adata.obs_names[sample_mask]

        # 提取原始 10x barcode（移除前缀与后缀）
        prefix = f"{sample_id}_"
        original_barcodes = []
        for cid in cell_ids:
            if not cid.startswith(prefix):
                warnings.warn(f"cell_id '{cid}' barcode 解析异常")
                original_barcodes.append(cid)
                continue
            rest = cid[len(prefix):]
            parts = rest.rsplit("-", 1)
            original_barcodes.append(parts[0] if len(parts) == 2 and parts[1].isdigit() else rest)

        # 定位 raw matrix
        raw_sample_dir = Path(raw_path) / sample_id / "raw_feature_bc_matrix"
        if not (raw_sample_dir / "matrix.mtx.gz").exists() and not (raw_sample_dir / "matrix.mtx").exists():
            alt_raw = Path(raw_path) / "raw_feature_bc_matrix"
            if (alt_raw / "matrix.mtx.gz").exists() or (alt_raw / "matrix.mtx").exists():
                raw_sample_dir = alt_raw
            else:
                print(f"  {sample_id}: raw_feature_bc_matrix 未找到，跳过")
                soupx_diagnostics[sample_id] = {"n_cells": n_cells, "status": "failed", "reason": "raw_feature_bc_matrix not found"}
                soupx_failed_samples.append(sample_id)
                soupx_needs_review = True
                continue

        try:
            # 导出 filtered matrix：用 layers["counts"]（原始 counts）而不是 X
            # X 在此前可能已被其他操作修改，layers["counts"] 才是唯一权威原始计数源
            sub_adata = adata[cell_ids].copy()
            filtered_export = os.path.join(soupx_tmp, f"{sample_id}_filtered")
            os.makedirs(filtered_export, exist_ok=True)
            count_mtx = sp.csr_matrix(sub_adata.layers["counts"]).T  # 从 counts layer 导出
            scipy.io.mmwrite(os.path.join(filtered_export, "matrix.mtx"), count_mtx)
            with open(os.path.join(filtered_export, "barcodes.tsv"), "w") as f:
                f.write("\n".join(original_barcodes) + "\n")
            with open(os.path.join(filtered_export, "features.tsv"), "w") as f:
                for gn in sub_adata.var_names:
                    f.write(f"{gn}\t{gn}\tGene Expression\n")

            work_dir = os.path.join(soupx_tmp, sample_id)
            cmd = [RSCRIPT_BIN, "--vanilla", soupx_script,
                   os.path.abspath(work_dir), os.path.abspath(filtered_export),
                   os.path.abspath(str(raw_sample_dir)), sample_id]
            print(f"  {sample_id}: 执行 SoupX...")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
            for line in result.stdout.strip().split("\n"):
                print(f"    [R] {line}")

            # 双重判定：R exit code + soupx_status.json
            # R 脚本失败时 quit(status=1) 且写 status=failed——两重信号都要尊重
            status_json_path = os.path.join(work_dir, "soupx_status.json")
            soupx_failed = False
            soupx_fail_reason = None

            _soupx_status = {}  # 初始化为空字典；若 json 文件存在则覆盖
            if not os.path.exists(status_json_path):
                soupx_failed = True
                soupx_fail_reason = "soupx_status.json not produced (R crash)"
            else:
                with open(status_json_path) as f:
                    _soupx_status = json.loads(f.read())
                if _soupx_status.get("status") != "success":
                    soupx_failed = True
                    soupx_fail_reason = _soupx_status.get("reason", f"status={_soupx_status.get('status')}")
                if result.returncode != 0:
                    soupx_failed = True
                    if not soupx_fail_reason:
                        soupx_fail_reason = f"R exit code={result.returncode}"

            if soupx_failed:
                print(f"  {sample_id}: SoupX 失败 ({soupx_fail_reason})，不写校正结果")
                soupx_needs_review = True
                soupx_failed_samples.append(sample_id)
                soupx_diagnostics[sample_id] = {
                    "n_cells": n_cells, "status": _soupx_status.get("status", "failed"),
                    "reason": soupx_fail_reason,
                    "rho": _soupx_status.get("rho_global"),
                }
                if result.stderr:
                    print(f"    [R stderr] {result.stderr[:300]}")
                # 清理 sub_adata 后 continue（跳过写 counts_soupx）
                del sub_adata
                continue

            # 成功路径：读校正矩阵 + 三重校验
            corrected_mtx_fp = os.path.join(work_dir, "corrected_counts.mtx")
            corrected = sp.csr_matrix(scipy.io.mmread(corrected_mtx_fp).T, dtype=np.float32)
            if corrected.shape != (n_cells, sub_adata.n_vars):
                print(f"  形状不匹配: corrected={corrected.shape} vs expected=({n_cells},{sub_adata.n_vars})，跳过")
                del sub_adata
                continue

            # barcode + gene order validation（R 输出契约对齐校验）
            with open(os.path.join(work_dir, "barcodes.tsv")) as f:
                if [l.strip() for l in f] != original_barcodes:
                    print(f"  barcode 顺序不匹配，跳过")
                    del sub_adata
                    continue
            out_features = pd.read_csv(os.path.join(work_dir, "features.tsv"), sep="\t", header=None)
            if out_features.iloc[:, 1].tolist() != sub_adata.var_names.tolist():
                print(f"  基因顺序不匹配，跳过")
                del sub_adata
                continue

            # 整数化（可选）：SoupX adjustCounts 输出非整数
            if SOUPX_INTEGER_ROUND:
                corrected.data = np.rint(corrected.data)
                corrected.eliminate_zeros()  # 取整可能产生显式零，清理节省内存

            # 写入 counts_soupx layer（整数位置索引直接写 CSR，防视图不写回）
            # 红线：绝不写 X 或 layers["counts"]
            cell_indices = np.where(sample_mask.values)[0]
            adata.layers["counts_soupx"][cell_indices, :] = corrected

            # 标记成功
            adata.obs.loc[cell_ids, "ambient_correction_applied"] = True
            n_soupx_corrected += n_cells
            print(f"  {sample_id}: SoupX 完成, {n_cells} 细胞已校正 → layers['counts_soupx']"
                  f"  (rho={_soupx_status.get('rho_global', 'N/A')})")

            # per-sample 诊断
            soupx_diagnostics[sample_id] = {
                "n_cells": n_cells,
                "status": "success",
                "rho": _soupx_status.get("rho_global"),
                "n_droplets": _soupx_status.get("n_droplets"),
                "n_cells_filtered": _soupx_status.get("n_cells_filtered"),
                "n_genes": _soupx_status.get("n_genes"),
                "n_common_genes": _soupx_status.get("n_common_genes"),
                "method_params": _soupx_status.get("method_params"),
            }

            del sub_adata
        except subprocess.TimeoutExpired:
            print(f"  {sample_id}: 超时（10min），跳过")
            soupx_failed_samples.append(sample_id)
            soupx_needs_review = True
            soupx_diagnostics[sample_id] = {"n_cells": n_cells, "status": "failed", "reason": "timeout"}
        except Exception as e:
            print(f"  {sample_id}: 异常 ({type(e).__name__}): {e}")
            soupx_failed_samples.append(sample_id)
            soupx_needs_review = True
            soupx_diagnostics[sample_id] = {"n_cells": n_cells, "status": "failed", "reason": f"{type(e).__name__}: {e}"}

    # --- 循环后收尾 ---
    soupx_applied = n_soupx_corrected > 0

    # 确定全局状态
    if soupx_failed_samples and n_soupx_corrected > 0:
        soupx_status_label = "partial"
    elif n_soupx_corrected > 0:
        soupx_status_label = "success"
    elif soupx_failed_samples:
        soupx_status_label = "failed"
    else:
        soupx_status_label = "skipped"

    adata.uns["soupx_contract"] = {
        "enabled": SOUPX_ENABLED,
        "status": soupx_status_label,
        "n_cells_corrected": n_soupx_corrected,
        "integer_round": SOUPX_INTEGER_ROUND,
        "per_sample": soupx_diagnostics,
        "failed_samples": soupx_failed_samples,
        "needs_review": soupx_needs_review,
    }

    # 更新 expression_contract.soupx_layer（决策3 核心 + 任务点1）
    # 红线：不改其他 7 字段，只更新 soupx_layer + append processing_history
    # 仅当真的写了 counts_soupx 时才更新——不谎报有校正层
    if n_soupx_corrected > 0:
        adata.uns["expression_contract"]["soupx_layer"] = "counts_soupx"
        adata.uns["expression_contract"]["processing_history"].append(
            "01_nancang: SoupX ambient correction → layers[counts_soupx], counts layer untouched"
        )

    print(f"\nSoupX 校正: {n_soupx_corrected} 细胞已校正（写入 layers['counts_soupx']）")
    print(f"expression_contract.soupx_layer = {adata.uns['expression_contract']['soupx_layer']}")
    if soupx_needs_review:
        print(f"[NEEDS_REVIEW] 失败样本: {soupx_failed_samples}")
    print(f"ambient_correction_applied: {adata.obs['ambient_correction_applied'].value_counts().to_dict()}")

    shutil.rmtree(soupx_tmp, ignore_errors=True)


## SoupX 校正后 QC 重算

**决策3**：SoupX 校正后必须基于校正后矩阵重算全部 QC 指标（total_counts / n_genes / pct_counts_mt），
再进行 QC 过滤与 doublet 检测。若仍用校正前指标判断细胞质量，等于白做 SoupX——因为高污染细胞的
原始指标可能误导过滤决策。

**重算源**：由 `expression_contract.soupx_layer` 单点决定（成功=`counts_soupx`，跳过/失败=`counts`）。
scanpy 1.11.5 的 `calculate_qc_metrics` 支持 `layer=` 参数指定计算源，无需动 X。

**列名对齐**：scanpy 默认输出 `n_genes_by_counts`，但 notebook 下游消费 `n_genes`，此处做别名映射。


In [ ]:
# SoupX 后重算 QC（决策3）：QC 指标必须基于校正后矩阵
# 若用校正前指标判断细胞质量 = 高污染细胞可能因"看起来还行"被误保留
# 重算源位置单点可见：expression_contract.soupx_layer
# 全线粒体基因标记基于大写基因名（F3 已保证统一大写）

_qc_source_layer = adata.uns["expression_contract"].get("soupx_layer")
_qc_source_layer = _qc_source_layer if _qc_source_layer in adata.layers else "counts"
qc_recomputed_source = _qc_source_layer  # 供 QC 报告 + checkpoint 记账

# 重建 var["mt"]（线粒体基因标记，前缀 MT- 大写）
# 注意：nancang 数据在 F3 已统一为全大写基因名
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
n_mt_genes = int(adata.var["mt"].sum())
print(f"线粒体基因标记: {n_mt_genes} 个 (MT- 前缀, 大写)")

# 基于校正后矩阵重算 QC 指标（scanpy 1.11.5 支持 layer= 参数，原生 CSR 不 densify）
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False,
                           inplace=True, layer=_qc_source_layer)
# 列名对齐：scanpy 默认 n_genes_by_counts → 下游消费 n_genes
adata.obs["n_genes"] = adata.obs["n_genes_by_counts"]

print(f"QC 指标已基于 {_qc_source_layer} 重算")
print(f"  total_counts: median={adata.obs['total_counts'].median():.0f}, "
      f"mean={adata.obs['total_counts'].mean():.0f}")
print(f"  n_genes: median={adata.obs['n_genes'].median():.0f}, "
      f"mean={adata.obs['n_genes'].mean():.0f}")
print(f"  pct_counts_mt: median={adata.obs['pct_counts_mt'].median():.2f}%, "
      f"mean={adata.obs['pct_counts_mt'].mean():.2f}%")

# 基因复杂度也基于重算后的指标更新
adata.obs["log_complexity"] = np.log10(adata.obs["n_genes"] + 1) / np.log10(adata.obs["total_counts"] + 1)
print(f"  log_complexity: median={adata.obs['log_complexity'].median():.4f}")


## 双细胞鉴定（Scrublet，决策8 三态定级）

**决策8**：三态定级（singlet/uncertain/doublet），只标记不物理删除细胞。
高置信 doublet 默认排除（`doublet_include=False`），uncertain 默认纳入（由 `DOUBLET_UNCERTAIN_INCLUDE` 控制）。
01 保存完整对象，02 按 `doublet_include` 构建整合对象。

双细胞（doublet）是两个细胞被误包在同一个液滴中测序。其基因表达是两种细胞类型的混合，
会干扰细胞类型注释和差异表达分析。

**处理策略**：使用 manifest-driven 跳过逻辑。
- 若 manifest `preprocessing_done` 含 `doublet_removal`→跳过（作者已处理）
- 若 manifest `qc_overrides.doublet_removal.skip=true`→跳过
- 否则按样本独立运行 scrublet

**注意**：doublet 仅标记不移除。PI 在查看下游聚类后可根据 doublet 是否形成独立小群来决定。

In [ ]:
# 双细胞鉴定（决策8）：三态定级 + per-sample scrublet
#
# 本节只做检测执行与 per-sample 三态定级（singlet/uncertain/doublet）。
# include 派生、needs_review 诊断、阈值持久化由后续 cell 完成。
# 决策3+8：Nancang 在 SoupX 校正 + 重算 QC 后检测 doublet；
# scrublet 输入层与重算 QC 同源（_qc_source_layer），逻辑自洽。
import scrublet as scr
# scrublet 版本号（用于 provenance 记录）
try:
    _scrublet_version = scr.__version__
except AttributeError:
    try:
        from importlib.metadata import version
        _scrublet_version = version("scrublet")
    except Exception:
        _scrublet_version = "unknown"

# ---- obs 列初始化（fresh-kernel 安全，不依赖残留状态）----
# 未检测样本/跳过场景所有列保留默认值，确保下游 02/测试不因列缺失报错
adata.obs["doublet_score"] = np.nan          # float；NaN 表示未检测
adata.obs["doublet_class"] = "singlet"       # 分类：singlet | uncertain | doublet；默认 singlet
adata.obs["predicted_doublet"] = False       # 向后兼容旧列：= doublet_class=="doublet"（仅高置信）
adata.obs["doublet_include"] = True          # 下游纳入控制；默认 True（只排除高置信 doublet）

# ---- manifest-driven skip check（保留既有逻辑）----
with open(MANIFEST_PATH) as f:
    manifest = yaml.safe_load(f)
pp_done = manifest.get("preprocessing_done", [])
qc_override = manifest.get("qc_overrides", {}).get("doublet_removal", {})

skip_doublet = False
skip_reason = None
if "doublet_removal" in pp_done:
    skip_doublet = True
    skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
elif qc_override.get("skip"):
    skip_doublet = True
    skip_reason = qc_override.get("reason", "qc_overrides.doublet_removal.skip=True")

# ---- per-sample 诊断容器 + 阈值记录 ----
doublet_diagnostics = {}         # per-sample 诊断（写入 uns["doublet_contract"]）
per_sample_thresholds = {}       # per-sample 阈值记录
doublet_needs_review = False     # 全局 needs_review 布尔（供 checkpoint cell 消费）
needs_review_reasons = []        # needs_review 原因列表

# ---- scrublet 输入层：与 SoupX 后 QC 重算同源（单点决定，打印可见）----
# 决策3+8：doublet 在 SoupX 校正后检测，输入与重算 QC 同源以保持逻辑自洽
_dbl_source_layer = adata.uns["expression_contract"].get("soupx_layer") or "counts"
print(f"doublet 检测输入层: layers['{_dbl_source_layer}']")

if skip_doublet:
    # 跳过检测：所有细胞保持默认 singlet/include=True
    print(f"双细胞鉴定已跳过: {skip_reason}")
    doublet_method = "skipped"
    doublet_method_version = "n/a"
elif EXPECTED_DOUBLET_RATE is None:
    # 未设定预期率 → 不运行 scrublet
    print("EXPECTED_DOUBLET_RATE=None，未运行 scrublet（所有细胞标记 singlet，include 全 True）")
    doublet_method = "not_run"
    doublet_method_version = "n/a"
else:
    # 运行 Scrublet per sample
    doublet_method = "scrublet"
    doublet_method_version = _scrublet_version
    print(f"运行 Scrublet per sample（expected_doublet_rate={EXPECTED_DOUBLET_RATE}，输入层={_dbl_source_layer}）...")

    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_sample = sample_mask.sum()

        # ---- 细胞数不足 → 记 needs_review，不静默跳过 ----
        if n_sample < DOUBLET_MIN_CELLS:
            reason = f"too_few_cells: {sample_id} ({n_sample} cells < {DOUBLET_MIN_CELLS})"
            needs_review_reasons.append(reason)
            doublet_needs_review = True
            print(f"  {sample_id}: {n_sample} 细胞（< {DOUBLET_MIN_CELLS}），记 needs_review，"
                  f"该样本全部标记 singlet")
            doublet_diagnostics[sample_id] = {
                "n_cells": n_sample, "n_singlet": n_sample, "n_uncertain": 0,
                "n_doublet": 0, "doublet_rate": 0.0, "uncertain_rate": 0.0,
                "threshold_low": None, "threshold_high": None,
                "auto_threshold": None,
                "score_p50": None, "score_p90": None, "score_p99": None,
                "needs_review": True, "reason": reason,
            }
            per_sample_thresholds[sample_id] = {"low": None, "high": None, "auto": None}
            continue

        # ---- 子集 copy + scrublet 检测 ----
        # 决策抉择：用校正后层（若可用）或原始 counts 层，与 QC 重算同源
        sub = adata[sample_mask].copy()
        scrub_input = sub.layers[_dbl_source_layer]
        scrub = scr.Scrublet(
            scrub_input, expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            random_state=RANDOM_SEED,
        )
        doublet_scores, _ = scrub.scrub_doublets()
        auto_threshold = scrub.threshold_

        # ---- 定阈值：high 优先用 DOUBLET_SCORE_HIGH，其次 DOUBLET_SCORE_THRESHOLD（旧手动），最后自动 ----
        # 向后兼容：DOUBLET_SCORE_THRESHOLD 非 None 时覆盖 high（并打印说明）
        if DOUBLET_SCORE_THRESHOLD is not None:
            threshold_high = DOUBLET_SCORE_THRESHOLD
            print(f"  使用手动 DOUBLET_SCORE_THRESHOLD={threshold_high}（覆盖自动阈值）")
        elif DOUBLET_SCORE_HIGH is not None:
            threshold_high = DOUBLET_SCORE_HIGH
        else:
            threshold_high = auto_threshold if auto_threshold is not None else 0.25
            if auto_threshold is None:
                _reason = f"scrublet_threshold_none: {sample_id}（auto_threshold=None，fallback 0.25）"
                needs_review_reasons.append(_reason)
                doublet_needs_review = True

        # low 优先用 DOUBLET_SCORE_LOW，否则 high*0.5 派生
        threshold_low = DOUBLET_SCORE_LOW if DOUBLET_SCORE_LOW is not None else threshold_high * 0.5

        # ---- 向量化三态定级（不逐行循环）----
        classes = np.full(n_sample, "singlet", dtype=object)
        classes[doublet_scores > threshold_high] = "doublet"
        classes[(doublet_scores >= threshold_low) & (doublet_scores <= threshold_high)] = "uncertain"

        # 写回 adata.obs
        adata.obs.loc[sample_mask, "doublet_score"] = doublet_scores
        adata.obs.loc[sample_mask, "doublet_class"] = classes

        # ---- 直方图可视化（三区标注：绿=singlet界，红=doublet界）----
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(doublet_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.7)
        ax.axvline(threshold_low, color="green", linestyle=":", linewidth=1.5,
                   label=f"singlet/uncertain={threshold_low:.3f}")
        ax.axvline(threshold_high, color="red", linestyle="--", linewidth=1.5,
                   label=f"uncertain/doublet={threshold_high:.3f}")
        if auto_threshold is not None and abs(auto_threshold - threshold_high) > 1e-6:
            ax.axvline(auto_threshold, color="orange", linestyle="-.", linewidth=1,
                       label=f"auto={auto_threshold:.3f}")
        ax.set_xlabel("Doublet Score")
        ax.set_ylabel("细胞数")
        ax.set_title(f"{sample_id}: Doublet Score 分布（singlet | uncertain | doublet）")
        ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()

        # ---- per-sample 诊断 ----
        n_dbl = int((classes == "doublet").sum())
        n_unc = int((classes == "uncertain").sum())
        n_sgl = int((classes == "singlet").sum())
        d_rate = n_dbl / n_sample
        u_rate = n_unc / n_sample

        score_p50 = round(float(np.percentile(doublet_scores, 50)), 4)
        score_p90 = round(float(np.percentile(doublet_scores, 90)), 4)
        score_p99 = round(float(np.percentile(doublet_scores, 99)), 4)

        # 样本级 needs_review 判定
        sample_needs_review = False
        sample_reasons = []
        if d_rate > DOUBLET_RATE_ALERT_HIGH:
            sample_needs_review = True
            sample_reasons.append(f"doublet_rate {d_rate:.1%} > alert {DOUBLET_RATE_ALERT_HIGH:.1%}")
        if DOUBLET_RATE_ALERT_LOW is not None and d_rate < DOUBLET_RATE_ALERT_LOW:
            sample_needs_review = True
            sample_reasons.append(f"doublet_rate {d_rate:.1%} < alert_low {DOUBLET_RATE_ALERT_LOW:.1%}")
        if sample_needs_review:
            doublet_needs_review = True
            needs_review_reasons.extend(
                [f"{sample_id}: {r}" for r in sample_reasons]
            )

        doublet_diagnostics[sample_id] = {
            "n_cells": n_sample, "n_singlet": n_sgl, "n_uncertain": n_unc,
            "n_doublet": n_dbl,
            "doublet_rate": round(d_rate, 4),
            "uncertain_rate": round(u_rate, 4),
            "threshold_low": round(threshold_low, 4),
            "threshold_high": round(threshold_high, 4),
            "auto_threshold": round(auto_threshold, 4) if auto_threshold is not None else None,
            "score_p50": score_p50, "score_p90": score_p90, "score_p99": score_p99,
            "needs_review": sample_needs_review,
            "reason": "; ".join(sample_reasons) if sample_reasons else None,
        }
        per_sample_thresholds[sample_id] = {
            "low": round(threshold_low, 4),
            "high": round(threshold_high, 4),
            "auto": round(auto_threshold, 4) if auto_threshold is not None else None,
        }

        status_flag = " [NEEDS_REVIEW]" if sample_needs_review else ""
        print(f"  {sample_id}: sgl={n_sgl} unc={n_unc} dbl={n_dbl} / {n_sample} "
              f"(dbl_rate={d_rate:.1%}, unc_rate={u_rate:.1%}){status_flag}")

print(f"\n检测完成：method={doublet_method}，"
      f"全局 needs_review={'是' if doublet_needs_review else '否'}，"
      f"输入层={_dbl_source_layer}")


## 双细胞三态收尾：include 派生 + 诊断持久化

决策8：只标记不物理删除细胞。`doublet_include` 列供下游 02_merged 构建整合对象时使用。
高置信 doublet 默认排除，uncertain 由 `DOUBLET_UNCERTAIN_INCLUDE` 控制。


In [ ]:
# 双细胞三态收尾：include 列派生 + needs_review 诊断 + 持久化到 uns["doublet_contract"]
#
# 决策8：只标记不物理删除细胞（01 保存完整对象；02 按 doublet_include 构建整合对象）

# ---- include 列派生（决策8：排除高置信 doublet；uncertain 由 DOUBLET_UNCERTAIN_INCLUDE 决定）----
# 重置所有为 True，再按规则排除
adata.obs["doublet_include"] = True
adata.obs.loc[adata.obs["doublet_class"] == "doublet", "doublet_include"] = False
if not DOUBLET_UNCERTAIN_INCLUDE:
    adata.obs.loc[adata.obs["doublet_class"] == "uncertain", "doublet_include"] = False

# 向后兼容：predicted_doublet = 仅高置信 doublet（旧列语义不变）
adata.obs["predicted_doublet"] = adata.obs["doublet_class"] == "doublet"

# ---- 全局汇总 ----
n_dbl = int((adata.obs["doublet_class"] == "doublet").sum())
n_unc = int((adata.obs["doublet_class"] == "uncertain").sum())
n_sgl = int((adata.obs["doublet_class"] == "singlet").sum())
n_excluded = int((~adata.obs["doublet_include"]).sum())

# 修 kim latent bug：nancang QC 报告 cell 消费 n_doublets_total
# kim 只定义 n_dbl，从不定义 n_doublets_total → fresh-kernel 下 NameError
# nancang 此处显式定义，保证 QC 报告 cell 不会报错
n_doublets_total = n_dbl  # 高置信 doublet 计数，供 QC 报告 cell（3a046d80）

print(f"双细胞三态汇总: singlet={n_sgl}  uncertain={n_unc}  doublet={n_dbl}")
print(f"  纳入下游={adata.n_obs - n_excluded}  排除={n_excluded}"
      f"（其中 high-conf doublet={n_dbl}"
      + (f", uncertain 排除={n_unc}" if not DOUBLET_UNCERTAIN_INCLUDE else "")
      + "）")

if doublet_needs_review:
    print(f"\n[NEEDS_REVIEW] 双细胞检测存在异常，需 PI 审阅 per-sample 诊断表：")
    for r in needs_review_reasons:
        print(f"  - {r}")
else:
    print("\n双细胞检测无异常。")

# ---- 每样本诊断表打印 ----
print("\n--- Per-sample 双细胞诊断 ---")
for sid in sorted(doublet_diagnostics.keys()):
    d = doublet_diagnostics[sid]
    nr_flag = " [NEEDS_REVIEW]" if d.get("needs_review") else ""
    print(f"  {sid}: {d['n_cells']} cells | "
          f"sgl={d['n_singlet']} unc={d['n_uncertain']} dbl={d['n_doublet']} | "
          f"dbl_rate={d['doublet_rate']:.1%} unc_rate={d['uncertain_rate']:.1%} | "
          f"thr=[{d['threshold_low']}, {d['threshold_high']}]{nr_flag}")
    if d.get("reason"):
        print(f"        reason: {d['reason']}")

# ---- 阈值与元数据持久化到 uns["doublet_contract"]（独立于 expression_contract）----
adata.uns["doublet_contract"] = {
    "method": doublet_method,
    "method_version": doublet_method_version,
    "per_sample_thresholds": per_sample_thresholds,
    "uncertain_include": DOUBLET_UNCERTAIN_INCLUDE,
    "rate_alert_high": DOUBLET_RATE_ALERT_HIGH,
    "rate_alert_low": DOUBLET_RATE_ALERT_LOW,
    "n_singlet": n_sgl,
    "n_uncertain": n_unc,
    "n_doublet": n_dbl,
    "n_excluded": n_excluded,
    "needs_review": bool(doublet_needs_review),
    "needs_review_reasons": needs_review_reasons,
    "per_sample_diagnostics": doublet_diagnostics,
    "random_seed": RANDOM_SEED,
}
print(f"\ndoublet_contract 已写入 uns（method={doublet_method}，"
      f"needs_review={bool(doublet_needs_review)}）")

## 细胞周期评分

使用 Tirosh et al. (2015) 的 S 期和 G2M 期 marker genes 对每个细胞打分。
细胞周期阶段（G1/S/G2M）是重要的技术协变量——如果不同样本/条件的细胞周期分布
不均衡，可能在差异表达分析中引入混淆。

In [ ]:
# 细胞周期评分（Tirosh 2015 marker genes）
if SCORE_CELL_CYCLE:
    s_genes = [
    # S 期标志基因（Tirosh et al. 2015）
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG", "GINS2", "MCM6",
    "CDCA7", "DTL", "PRIM1", "UHRF1", "MLF1IP", "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1",
    "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2",
    "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2", "USP1", "CLSPN", "POLA1",
    "CHAF1B", "BRIP1", "E2F8",
]
    g2m_genes = [
    # G2M 期标志基因
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", "CKS2", "NUF2",
    "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2",
    "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP", "CDCA3", "HN1",
    "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2",
    "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF", "NEK2",
    "G2E3", "GAS2L3", "CBX5", "CENPA",
]
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
    print("细胞周期评分完成")
    print(adata.obs["phase"].value_counts())
else:
    print("SCORE_CELL_CYCLE=False，跳过")


## 过滤

根据上一步得出的阈值过滤低质量细胞。

**不在此步移除的**：
- 双细胞（predicted_doublet）：仅标记，供下游聚类后决定
- 血红蛋白高表达细胞：仅标记（flag_hb）
- 应激高表达细胞：仅标记

In [ ]:
# 过滤：应用 QC 阈值
print("===== QC 过滤 =====")
cells_before = adata.n_obs
print(f"过滤前细胞数: {cells_before:,}")

if PER_SAMPLE_MAD:
    # Per-sample 模式：按每个 sample 的阈值独立判断
    import warnings

    _n_samples = adata.obs["sample_id"].nunique()
    if _n_samples < 2:
        warnings.warn(
            f"检测到当前数据集仅包含 {_n_samples} 个样本（sample_id 唯一值 = {_n_samples}），"
            f"但 PER_SAMPLE_MAD = True。per-sample MAD 在单样本下等价于全局 MAD，"
            f"不再具备每个样本独立基线的统计学优势。"
            f"建议：将 PER_SAMPLE_MAD 设为 False 以明确意图；"
            f"或确认数据集确有多样本后再运行。"
            f"当前管线将以等效全局 MAD 模式继续运行。",
            UserWarning,
        )
    keep = pd.Series(True, index=adata.obs_names)
    filter_counts = {"n_genes_lower": 0, "n_genes_upper": 0, "total_counts": 0, "pct_counts_mt": 0}
    for sample_id in adata.obs["sample_id"].unique():
        smask = adata.obs["sample_id"] == sample_id
        st = thresholds["n_genes"]["per_sample"].get(sample_id)
        if st and st.get("lower") is not None:
            n_fail = ((adata.obs.loc[smask, "n_genes"] < st["lower"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "n_genes"] >= st["lower"]
            filter_counts["n_genes_lower"] += n_fail
        if st and st.get("upper") is not None:
            n_fail = ((adata.obs.loc[smask, "n_genes"] > st["upper"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "n_genes"] <= st["upper"]
            filter_counts["n_genes_upper"] += n_fail
        st_tc = thresholds["total_counts"]["per_sample"].get(sample_id)
        if st_tc and st_tc.get("lower") is not None:
            n_fail = ((adata.obs.loc[smask, "total_counts"] < st_tc["lower"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "total_counts"] >= st_tc["lower"]
            filter_counts["total_counts"] += n_fail
        st_mt = thresholds["pct_counts_mt"]["per_sample"].get(sample_id)
        if st_mt and st_mt.get("upper") is not None:
            n_fail = ((adata.obs.loc[smask, "pct_counts_mt"] > st_mt["upper"]) & keep.loc[smask]).sum()
            keep.loc[smask] &= adata.obs.loc[smask, "pct_counts_mt"] <= st_mt["upper"]
            filter_counts["pct_counts_mt"] += n_fail
    print(f"  n_genes 偏低: {filter_counts['n_genes_lower']:,}  偏高: {filter_counts['n_genes_upper']:,}")
    print(f"  total_counts 偏低: {filter_counts['total_counts']:,}")
    print(f"  pct_counts_mt 偏高: {filter_counts['pct_counts_mt']:,}")
else:
    keep = pd.Series(True, index=adata.obs_names)
    if thresholds["n_genes"]["lower"] is not None:
        n_fail = (adata.obs["n_genes"] < thresholds["n_genes"]["lower"]).sum()
        keep &= adata.obs["n_genes"] >= thresholds["n_genes"]["lower"]
        print(f"  n_genes < {thresholds['n_genes']['lower']:.0f}: {n_fail:,} 失败")
    if thresholds["n_genes"]["upper"] is not None:
        n_fail = (adata.obs["n_genes"] > thresholds["n_genes"]["upper"]).sum()
        keep &= adata.obs["n_genes"] <= thresholds["n_genes"]["upper"]
        print(f"  n_genes > {thresholds['n_genes']['upper']:.0f}: {n_fail:,} 失败")
    if thresholds["total_counts"]["lower"] is not None:
        n_fail = (adata.obs["total_counts"] < thresholds["total_counts"]["lower"]).sum()
        keep &= adata.obs["total_counts"] >= thresholds["total_counts"]["lower"]
        print(f"  total_counts < {thresholds['total_counts']['lower']:.0f}: {n_fail:,} 失败")
    if thresholds["pct_counts_mt"]["upper"] is not None:
        n_fail = (adata.obs["pct_counts_mt"] > thresholds["pct_counts_mt"]["upper"]).sum()
        keep &= adata.obs["pct_counts_mt"] <= thresholds["pct_counts_mt"]["upper"]
        print(f"  pct_counts_mt > {thresholds['pct_counts_mt']['upper']:.1f}%: {n_fail:,} 失败")

print(f"\n保留细胞: {keep.sum():,} / {cells_before:,} ({100*keep.sum()/cells_before:.1f}%)")

obs_pre_filter = adata.obs.copy()  # 内存安全：只复制 obs DataFrame，不复制矩阵
adata = adata[keep].copy()
cells_after = adata.n_obs
print(f"去除细胞数: {cells_before - cells_after:,} ({100*(cells_before-cells_after)/cells_before:.1f}%)")

# 基因过滤：移除仅在极少细胞中检测到的噪声基因
n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
n_genes_after = adata.n_vars
print(f"基因过滤：{n_genes_before:,} → {n_genes_after:,}（移除 {n_genes_before - n_genes_after:,} 个仅在 <{MIN_CELLS_PER_GENE} 细胞中检测到的基因）")

## 过滤交叉诊断

MAD 过滤与各标记物（doublet/HB/stress）的关系：被过滤掉的细胞是否富集了这些标记？


In [ ]:
# === 过滤交叉诊断：MAD 过滤与各标记物的关系 ===
n_removed_total = cells_before - cells_after
if n_removed_total > 0 and 'obs_pre_filter' in dir():
    print("===== 被过滤细胞的标记物富集分析 =====")
    _removed_idx = obs_pre_filter.index.difference(adata.obs_names)
    _kept_idx = adata.obs_names
    
    enrichment = {}
    if "predicted_doublet" in obs_pre_filter.columns:
        dbl_removed = obs_pre_filter.loc[_removed_idx, "predicted_doublet"].mean()
        dbl_kept = obs_pre_filter.loc[_kept_idx, "predicted_doublet"].mean()
        enrichment["doublet"] = {
            "在被过滤细胞中": f"{dbl_removed:.1%}",
            "在保留细胞中": f"{dbl_kept:.1%}",
            "富集倍数": round(dbl_removed / max(dbl_kept, 0.001), 1)
        }
    if "flag_hb" in obs_pre_filter.columns:
        hb_removed = obs_pre_filter.loc[_removed_idx, "flag_hb"].mean()
        hb_kept = obs_pre_filter.loc[_kept_idx, "flag_hb"].mean()
        enrichment["hemoglobin"] = {
            "在被过滤细胞中": f"{hb_removed:.1%}",
            "在保留细胞中": f"{hb_kept:.1%}",
            "富集倍数": round(hb_removed / max(hb_kept, 0.001), 1)
        }
    if "pct_counts_stress" in obs_pre_filter.columns:
        stress_removed = obs_pre_filter.loc[_removed_idx, "pct_counts_stress"].median()
        stress_kept = obs_pre_filter.loc[_kept_idx, "pct_counts_stress"].median()
        enrichment["stress(median_pct)"] = {
            "在被过滤细胞中": f"{stress_removed:.2f}%",
            "在保留细胞中": f"{stress_kept:.2f}%",
            "富集倍数": round(stress_removed / max(stress_kept, 0.001), 1)
        }
    if "log_complexity" in obs_pre_filter.columns:
        cx_removed = obs_pre_filter.loc[_removed_idx, "log_complexity"].median()
        cx_kept = obs_pre_filter.loc[_kept_idx, "log_complexity"].median()
        enrichment["complexity(median)"] = {
            "在被过滤细胞中": f"{cx_removed:.3f}",
            "在保留细胞中": f"{cx_kept:.3f}",
            "富集倍数": "N/A"
        }
    
    if enrichment:
        display(pd.DataFrame(enrichment).T)
        print("\n解读：富集倍数 > 3 = MAD 过滤已隐含覆盖该标记物；≈ 1 = 两者独立，下游需额外处理")
    
    del obs_pre_filter  # 释放内存
else:
    print("无细胞被过滤，跳过交叉诊断")


## Per-sample 过滤影响


In [ ]:
# === Per-sample 过滤影响表 ===
if PER_SAMPLE_MAD and "filter_per_sample_stats" not in dir():
    pass  # 已通过 per-sample 阈值实现，此处汇总过滤后细胞分布

print("===== Per-sample 过滤影响 =====")
if "qc_report_v1" in adata.uns:
    print(f"总计: {adata.uns['qc_report_v1']['cells_before']:,} -> {adata.uns['qc_report_v1']['cells_after']:,} "
          f"(去除 {adata.uns['qc_report_v1']['pct_removed']}%)")
# 每个 sample 的细胞数（过滤后）
sample_counts = adata.obs["sample_id"].value_counts().sort_index()
print(sample_counts.to_string())
total = sample_counts.sum()
print(f"\n各 sample 占比:")
for sid, n in sample_counts.items():
    print(f"  {sid}: {n:,} ({100*n/total:.1f}%)")
min_sample = sample_counts.idxmin()
max_sample = sample_counts.idxmax()
if sample_counts.max() / max(sample_counts.min(), 1) > 5:
    print(f"\n⚠️ 样本间细胞数差异 > 5 倍（{max_sample}={sample_counts.max()} vs {min_sample}={sample_counts.min()}）")
    print("  -> 下游整合时可能需要考虑下采样平衡（02_merged 的 DOWNSAMPLE_TO_MIN）")


## 过滤前后对比

复刻过滤前的 QC 图，供 PI 做直观对比。

**对比检查要点**：
- 小提琴图：各指标的分布尾部是否被正确截断
- 散点图：被移除的细胞是否集中在预期区域（低基因数 + 高 MT% 区）
- 剩余细胞数是否合理：通常保留 80-95%

In [ ]:
# 过滤后 QC 小提琴图
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="sample_id", rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤后）")
plt.tight_layout()
fig.savefig("results/figures/01_nancang_qc_violin_post.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt", ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤后）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts", ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤后）")
plt.tight_layout()
fig.savefig("results/figures/01_nancang_qc_scatter_post.png", dpi=150, bbox_inches="tight")
plt.show()


## QC 报告摘要

汇总本次 QC 的全部参数与结果，写入 `adata.uns["qc_report_v1"]`，供下游 notebook 读取。

In [ ]:
# QC 报告摘要
n_removed = cells_before - cells_after
qc_report = {
    "strategy": QC_STRATEGY,
    "n_mad": N_MAD if QC_STRATEGY == "adaptive" else None,
    "thresholds": {k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in thresholds.items()},
    "cells_before": int(cells_before),
    "cells_after": int(cells_after),
    "cells_removed": int(n_removed),
    "pct_removed": round(100 * n_removed / cells_before, 1) if cells_before > 0 else 0,
    "pre_qc_stats": qc_pre_stats,
    # --- SoupX 记账（决策3）---
    "soupx_applied": soupx_applied,
    "soupx_cells_corrected": n_soupx_corrected,
    "soupx_status": adata.uns.get("soupx_contract", {}).get("status"),
    "soupx_needs_review": globals().get("soupx_needs_review", False),
    "qc_recomputed_source": globals().get("qc_recomputed_source"),
    # --- doublet 三态记账（决策8）---
    # 用 globals().get guard 防 shared test 合成对象报 NameError
    "doublet_rate_pct": round(100 * globals().get("n_dbl", 0) / cells_before, 2) if cells_before > 0 else 0,
    "n_singlet": globals().get("n_sgl", 0),
    "n_uncertain": globals().get("n_unc", 0),
    "n_doublet": globals().get("n_dbl", 0),
    "n_excluded": globals().get("n_excluded", 0),
    "doublet_needs_review": globals().get("doublet_needs_review", False),
    # --- 其他 ---
    "cell_cycle_scored": SCORE_CELL_CYCLE,
    "flag_hb": FLAG_HEMOGLOBIN if 'FLAG_HEMOGLOBIN' in dir() else True,
    "n_hb_flagged": n_hb_flagged if 'n_hb_flagged' in dir() else 0,
}
adata.uns["qc_report_v1"] = qc_report
print("===== QC 报告摘要 =====")
for k, v in qc_report.items():
    print(f"  {k}: {v}")


In [ ]:
# Checkpoint：写入 per-dataset h5ad
# 先计算所有可见门禁；只有到保存阶段才占用 RUN_ID。

# F3修复：基因 ID 轴统一性断言——检查 per-dataset 内基因名无大小写混用
# 过大写/小写混用在 merge 时会导致 inner join 基因交集意外坍塌
_gene_names = list(adata.var_names)
_upper_count = sum(1 for g in _gene_names if g[0].isupper()) if _gene_names else 0
_lower_count = sum(1 for g in _gene_names if g[0].islower()) if _gene_names else 0
_total = len(_gene_names)
if _upper_count > 0 and _lower_count > 0:
    raise ValueError(
        f"基因 ID 轴不一致：{_upper_count} 个大写首字母基因 + {_lower_count} 个小写首字母基因 共 {_total} 个。"
        f"请统一基因名大小写（如全部 .str.upper()）后再进入 merge。"
    )
print(f"基因 ID 轴一致性检查通过：{_total} 个基因，统一为{'大写' if _upper_count > 0 else '小写'}首字母")

# ---- expression_contract 硬门禁（决策1/2）：契约完整性 + layers["counts"] 存在性 ----
# 若 validate_expression_contract 抛 KeyError/ValueError → 契约不合格，stage FAILED
_contract_ok = True
_contract_error = None
try:
    _contract = validate_expression_contract(adata, expected_scale="raw_counts", stage="01")
except (KeyError, ValueError) as _e:
    _contract_ok = False
    _contract_error = str(_e)
# layers["counts"] 必须存在且为 CSR float32（与 expression_contract 中 counts_layer 字段一致）
_layers_counts_ok = (
    "counts" in adata.layers
    and sp.issparse(adata.layers["counts"])
    and adata.layers["counts"].dtype == np.float32
)

# ---- counts_layer_unchanged 门禁（决策3 红线核心）----
# 验证 layers["counts"] 未被 SoupX / 任何中间操作改动
# 与 expression_contract cell 中保存的 checksum 比对：sum + nnz 双重验证
# _counts_checksum / _counts_checksum_nnz 由 expression_contract cell (072ef661) 在 SoupX 前定义
_counts_layer_unchanged = True
if globals().get("_counts_checksum") is not None and "counts" in adata.layers:
    _curr_sum = float(adata.layers["counts"].sum())
    _curr_nnz = int(adata.layers["counts"].nnz)
    _counts_layer_unchanged = (
        abs(_curr_sum - globals().get("_counts_checksum", _curr_sum)) < 1
        and _curr_nnz == globals().get("_counts_checksum_nnz", _curr_nnz)
    )
    if not _counts_layer_unchanged:
        print(f"CRITICAL layers['counts'] 已被修改: "
              f"sum {globals().get('_counts_checksum')} → {_curr_sum}, "
              f"nnz {globals().get('_counts_checksum_nnz')} → {_curr_nnz}")

# ---- soupx_layer_consistent：契约声称与实际 layer 存在性必须一致 ----
_soupx_layer_consistent = True
_soupx_layer = adata.uns["expression_contract"].get("soupx_layer")
if _soupx_layer == "counts_soupx":
    _soupx_layer_consistent = (
        "counts_soupx" in adata.layers
        and adata.layers["counts_soupx"].dtype == np.float32
    )
# 若 soupx_layer 为 None 但 counts_soupx 存在，也接受（可能是旧行为迁移期）

_source_values = sorted(map(str, adata.obs["source_dataset"].dropna().unique())) if "source_dataset" in adata.obs.columns else []
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_sparse_float32": sp.issparse(adata.X) and adata.X.dtype == np.float32,
    "expression_contract": _contract_ok,
    "layers_counts_csr_f32": _layers_counts_ok,
    # 决策3 红线：counts 层未被改动
    "counts_layer_unchanged": _counts_layer_unchanged,
    # 决策3：soupx_layer 契约与实际一致
    "soupx_layer_consistent": _soupx_layer_consistent,
    "source_dataset_unique": len(_source_values) == 1 and not adata.obs["source_dataset"].isna().any(),
    "source_matches_manifest": _source_values == [str(source_dataset)],
    # ---- doublet 三态 postconditions（_hd guard，决策8）----
    # 仅在 doublet 检测已运行（doublet_class 列存在）时才验证 doublet postconditions
    # 未运行 doublet 的场景（如 shared test 合成 adata）doublet 相关键设为 True 跳过
}
_hd = "doublet_class" in adata.obs.columns
hard_postconditions["doublet_columns_present"] = all(
    c in adata.obs for c in ["doublet_score", "doublet_class", "doublet_include", "predicted_doublet"]
) if _hd else True
hard_postconditions["doublet_contract_present"] = "doublet_contract" in adata.uns if _hd else True
hard_postconditions["doublet_class_valid"] = (
    set(adata.obs["doublet_class"].unique()) <= {"singlet", "uncertain", "doublet"}
) if _hd else True
hard_postconditions["doublet_include_consistent"] = (
    not adata.obs.loc[adata.obs["doublet_class"] == "doublet", "doublet_include"].any()
) if _hd else True

if not _contract_ok:
    print(f"WARNING expression_contract 校验失败: {_contract_error}")

# ---- needs_review 判定（决策3 + 决策8）----
# SoupX 启用但有失败样本 → NEEDS_REVIEW（只写 draft 不提升，交 PI 裁决）
# opus 裁决：SoupX 失败不进 FAILED（FAILED 语义是 hard postcondition 崩，
# SoupX 失败是"计算完成但需 PI 决策是否接受未校正继续"）
_doublet_needs_review = bool(globals().get("doublet_needs_review", False))
_soupx_needs_review = bool(globals().get("soupx_needs_review", False))
_needs_review = _doublet_needs_review or _soupx_needs_review

stage_status = determine_stage_status({}, hard_postconditions, needs_review=_needs_review, allow_no_required_methods=True)
effective_parameters = snapshot_effective_parameters(globals(), exclude=("RSCRIPT_BIN", "R_AVAILABLE"), path_root=Path(_root))
runtime_provenance = collect_runtime_provenance(_root, ("anndata", "scanpy", "numpy", "pandas", "scipy"))
manifest_sha256 = sha256_file(MANIFEST_PATH)
run_paths = prepare_run(RUN_ROOT, RUN_ID)
manifest_payload = {
    "run_id": RUN_ID, "stage": "01_qcd", "stage_status": stage_status.value,
    "source_dataset": str(source_dataset),
    "inputs": [{"path": MANIFEST_PATH, "sha256": manifest_sha256}],
    "effective_parameters": effective_parameters, "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions,
}
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")
adata.uns["stage"] = "01_qcd"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [MANIFEST_PATH]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = RUN_ID
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)

# ---- 分支处理 ----
if stage_status.value == "NEEDS_REVIEW":
    # SoupX 或 doublet 异常，只写 draft checkpoint，不提升为正式 checkpoint
    # 写入诊断摘要供 PI 审视
    if _hd:
        manifest_payload["doublet_summary"] = {
            "needs_review": _doublet_needs_review,
            "n_excluded": int((~adata.obs["doublet_include"]).sum()),
            "n_doublet": int((adata.obs["doublet_class"] == "doublet").sum()),
            "n_uncertain": int((adata.obs["doublet_class"] == "uncertain").sum()),
            "n_singlet": int((adata.obs["doublet_class"] == "singlet").sum()),
            "reasons": globals().get("needs_review_reasons", []),
        }
    # SoupX 诊断摘要
    _sc = adata.uns.get("soupx_contract", {})
    if _soupx_needs_review or _sc.get("needs_review"):
        manifest_payload["soupx_summary"] = {
            "status": _sc.get("status"),
            "n_cells_corrected": _sc.get("n_cells_corrected", 0),
            "failed_samples": _sc.get("failed_samples", []),
            "needs_review": bool(_sc.get("needs_review", False)),
        }
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    OUTPUT_PATH = str(run_paths.draft_dir / OUTPUT_FILENAME)
    print(f"[NEEDS_REVIEW] draft checkpoint 已写入 {OUTPUT_PATH}，未提升正式 checkpoint")
    reasons = []
    if _soupx_needs_review:
        reasons.append(f"SoupX 部分/全部样本失败: {_sc.get('failed_samples', [])}")
    if _doublet_needs_review:
        reasons.append(f"doublet 检测异常: {globals().get('needs_review_reasons', [])}")
    for r in reasons:
        print(f"  - {r}")
    print(f"  请 PI 审阅诊断表后再决定是否手动 promote 或调整参数重跑。")
elif stage_status.value == "FAILED":
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")
else:
    OUTPUT_PATH = str(promote_run(run_paths))
    print(f"OK 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")


# per_dataset schema 校验
from scrna_integration.per_dataset_schema import validate_per_dataset_output
_schema_result = validate_per_dataset_output(adata)
if not _schema_result["passed"]:
    print("per_dataset schema 校验 FAILED:")
    for e in _schema_result["errors"]:
        print(f"  [ERROR] {e}")
else:
    print("per_dataset schema 校验 PASSED")
if _schema_result["warnings"]:
    for w in _schema_result["warnings"]:
        print(f"  [WARN] {w}")
gc.collect()
del adata; gc.collect()
print("内存已释放。")


### Stage 01 Verdict

Nancang（活检组织，10x mtx 格式）QC 阶段完成。请逐项确认：

- [ ] 基因 ID 体系已同步（symbol → ensembl_id），大小写一致，无混用
- [ ] QC 阈值已确认合理（对照 N_MAD 敏感度曲线，N_MAD=5 适合活检组织数据）
- [ ] SoupX 环境 RNA 校正已完成（per-sample 校正结果检查，无失败样本）
- [ ] SoupX 校正后 QC 指标已重算（total_counts / n_genes / pct_mt 基于校正后矩阵）
- [ ] 双细胞检测结果已检查（per-sample 三态诊断表，高置信 doublet 已标记）
- [ ] 血红蛋白/应激基因标记已检查（仅标记不排除，供下游参考）
- [ ] 基因复杂度分布已检查（无异常低复杂度细胞批量污染）
- [ ] QC 过滤前后对比已确认（细胞保留率在预期范围，被移除细胞集中在低基因数+高 MT% 区域）
- [ ] 输出 h5ad 已写入 `results/runs/`，`validate_per_dataset_output()` schema 校验通过

**下一步**：全部 4 个 per-dataset 01 notebook 处理完成后，运行 `02_merged.ipynb` 进行跨数据集合并。